# MiniMax-H3-Image (G4 / H100 / A100 · SGLang Diffusion 6-Step PDD Native 4K T2I & 1–16 Ref I2I)

Serves `MiniMaxAI/MiniMax-H3-Image` (33.1B DiT + 33.4B Qwen3-VL encoder + MiniMaxH3ImageVAE, 6-step fused PDD schedule, online AdaLN precomputation) on a Colab G4 GPU by default (`--quantization fp8-all` resident ~50.4 GiB VRAM, 2.34s@2K / 14.45s@4K, or `--quantization bf16` native lossless ~92.2 GiB VRAM) via SGLang Diffusion with fail-closed Bearer token auth, network-friendly async `/api/ct/jobs` + content-addressed binary uploads + resumable `Range` downloads, OpenAI `/v1/images/*` compatibility, and zero-transfer Google Drive I/O.

Run it with the service script (`colabllm <name> up`), which creates the Colab session, mounts Drive with `colab dfs`, optionally creates a named Cloudflare tunnel, and injects `CT_*` settings. Cell order is the standard contract -- see `.agents/skills/colab-llm-service/SKILL.md`.

In [ ]:
# @title 0. Config -- values come from `colab exec --env` (defaults allow running in the Colab UI)
import json, os, re, secrets, shutil, stat, subprocess, sys, time, urllib.request, urllib.error

SERVICE      = os.environ.get("CT_SERVICE", "minimax-h3-image")
SERVICE_PORT = int(os.environ.get("CT_SERVICE_PORT", "8092"))
AUTH         = os.environ.get("CT_AUTH", "builtin")            # caddy | builtin | none
PUBLIC_PORT  = int(os.environ.get("CT_PUBLIC_PORT", "8092"))  # what cloudflared targets
AUTH_USER    = os.environ.get("CT_AUTH_USER", "admin")
AUTH_PASS    = os.environ.get("CT_AUTH_PASS") or secrets.token_urlsafe(18)
API_TOKEN    = os.environ.get("CT_API_TOKEN") or "sk-" + secrets.token_urlsafe(32)
TUNNEL_TOKEN = os.environ.get("CT_TUNNEL_TOKEN", "")            # set -> fixed domain (named tunnel)
PUBLIC_HOST  = os.environ.get("CT_PUBLIC_HOSTNAME", "")
CREDS_FROM_ENV = bool(os.environ.get("CT_AUTH_PASS"))

WORK = "/content/ct"                                            # logs + binaries of this notebook
os.makedirs(WORK, exist_ok=True)
for _p in ("/content/drive/MyDrive/colab-llm/hf_token.txt",
           "/content/drive/MyDrive/colab-services/hf_token.txt",
           "/content/drive/MyDrive/colab-services/qwen-flash/hf_token.txt"):
    if not os.environ.get("HF_TOKEN") and os.path.isfile(_p):
        _tok = open(_p).read().strip()
        if _tok:
            os.environ["HF_TOKEN"] = os.environ["HUGGING_FACE_HUB_TOKEN"] = _tok
            break
MOUNT_DRIVE = os.environ.get("CT_MOUNT_DRIVE", "1") != "0"
DATA_DIR = os.environ.get("CT_DATA_DIR", f"/content/drive/MyDrive/colab-llm/{SERVICE}")
if not MOUNT_DRIVE or not os.path.isdir("/content/drive/MyDrive"):   # Drive not mounted -> local fallback
    if DATA_DIR.startswith("/content/drive/"):
        DATA_DIR = f"/content/data/{SERVICE}"
os.makedirs(DATA_DIR, exist_ok=True)
PROCS = {}                                                      # name -> (Popen, restart function)

def ct_spawn(name, args, env=None, cwd=None):
    """Start a background process; output goes to /content/ct/<name>.log."""
    with open(f"{WORK}/{name}.log", "a") as logf:   # child keeps its own dup of the fd
        return subprocess.Popen(args, stdout=logf, stderr=subprocess.STDOUT, cwd=cwd,
                                env={**os.environ, **(env or {})}, start_new_session=True)

def ct_stop_proc(proc, timeout=15):
    """Terminate a ct_spawn process group, escalating to SIGKILL after a bounded wait."""
    import signal
    if proc is None or proc.poll() is not None:
        return
    for sig in (signal.SIGTERM, signal.SIGKILL):
        try:
            os.killpg(proc.pid, sig)
        except (ProcessLookupError, PermissionError):
            try:
                proc.send_signal(sig)
            except ProcessLookupError:
                return
        try:
            proc.wait(timeout=timeout)
            return
        except subprocess.TimeoutExpired:
            continue

def ct_gpu_stat(query="memory.used,memory.total,utilization.gpu"):
    """nvidia-smi summary that never raises (CPU runtimes / transient driver errors)."""
    try:
        return subprocess.check_output(["nvidia-smi", f"--query-gpu={query}", "--format=csv,noheader"],
                                       text=True, stderr=subprocess.DEVNULL, timeout=20).strip()
    except (OSError, subprocess.SubprocessError):
        return "cpu/no-gpu"

def ct_log_size(name):
    p = f"{WORK}/{name}.log"
    return os.path.getsize(p) if os.path.exists(p) else 0

def ct_log_since(name, offset):
    p = f"{WORK}/{name}.log"
    if not os.path.exists(p):
        return ""
    with open(p, "rb") as handle:
        handle.seek(min(offset, os.path.getsize(p)))
        return handle.read().decode("utf-8", errors="replace")

def ct_public_gate(public_url, probe, verify_auth, log, timeout=120, interval=8):
    """Wait for the public endpoint; auth failures are fatal, network errors are retried."""
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        try:
            code = probe()
        except Exception as err:   # DNS / edge propagation / TLS warm-up
            log(f"Public probe pending ({type(err).__name__}: {err})")
            time.sleep(interval)
            continue
        if code == 200:
            verify_auth()            # RuntimeError propagates: never report CT_READY with broken auth
            return
        log(f"Public probe pending (HTTP {code})")
        time.sleep(interval)
    raise RuntimeError(f"Public endpoint {public_url} failed readiness verification within {timeout}s")

def ct_watchdog(health, start_service, log, *, service_key="service", log_name="service",
                interval=60, grace=900, dead_markers=(), heartbeat=None):
    """Supervise PROCS forever: restart dead processes, debounce health failures after reloads."""
    started = time.monotonic()
    failures = 0
    grace_until = time.monotonic()
    log_offset = ct_log_size(log_name)

    def restart_service(reason):
        nonlocal failures, grace_until, log_offset
        log(f"[watchdog] {reason}; restarting {service_key}")
        ct_stop_proc(PROCS[service_key][0])
        PROCS[service_key] = (start_service(), start_service)
        failures = 0
        grace_until = time.monotonic() + grace
        log_offset = ct_log_size(log_name)

    while True:
        time.sleep(interval)
        for name, (proc, restart) in list(PROCS.items()):
            if proc.poll() is not None:
                log(f"[watchdog] {name} exited with {proc.returncode}; restarting\n{ct_tail(name, 20)}")
                PROCS[name] = (restart(), restart)
                if name == service_key:
                    failures = 0
                    grace_until = time.monotonic() + grace
                    log_offset = ct_log_size(log_name)
        if dead_markers:
            fresh = ct_log_since(log_name, log_offset)
            log_offset += len(fresh.encode("utf-8", errors="replace"))
            if any(marker in fresh for marker in dead_markers):
                restart_service("Detected dead worker marker in service log")
                continue
        try:
            healthy = bool(health())
        except Exception:
            healthy = False
        if healthy:
            failures = 0
        elif time.monotonic() >= grace_until:
            failures += 1
        if failures >= 6:
            restart_service("6 consecutive failed health checks")
        if heartbeat is not None:
            try:
                heartbeat(int(time.monotonic() - started))
            except Exception as err:
                log(f"[watchdog] heartbeat warning: {err}")

def ct_wait_http(url, timeout=600, ok=lambda code: code < 500):
    """Wait until url answers with a non-5xx status (401/404 count as 'listening')."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=5) as r:
                if ok(r.status): return r.status
        except urllib.error.HTTPError as e:
            if ok(e.code): return e.code
        except Exception:
            pass
        time.sleep(2)
    raise TimeoutError(f"{url} not ready after {timeout}s")

def ct_tail(name, n=30):
    p = f"{WORK}/{name}.log"
    return "".join(open(p, errors="replace").readlines()[-n:]) if os.path.exists(p) else ""

def ct_fetch_binary(path, url):
    if not os.path.exists(path):
        partial = f"{path}.{os.getpid()}.partial"
        with urllib.request.urlopen(url, timeout=60) as src, open(partial, "wb") as dst:
            shutil.copyfileobj(src, dst)
        os.chmod(partial, os.stat(partial).st_mode | stat.S_IEXEC)
        os.replace(partial, path)
    return path

print(f"service={SERVICE} port={SERVICE_PORT} auth={AUTH} public_port={PUBLIC_PORT} "
      f"tunnel={'named ' + PUBLIC_HOST if TUNNEL_TOKEN else 'quick'} data={DATA_DIR}")


In [ ]:
# @title 1. Setup -- SGLang Diffusion (`MiniMax-H3-Image`), local NVMe weights & compiler caches
import hashlib
import shutil
from pathlib import Path

def h3log(msg):
    print(f"[minimax-h3-image {time.strftime('%H:%M:%S')}] {msg}", flush=True)

MODEL_REPO = (os.environ.get("CT_H3_IMAGE_MODEL") or "MiniMaxAI/MiniMax-H3-Image").strip()
MODEL_ALIAS = (os.environ.get("CT_H3_IMAGE_ALIAS") or "minimax-h3-image").strip()
QUANTIZATION = (os.environ.get("CT_H3_IMAGE_QUANTIZATION") or "fp8-all").strip().lower()
PLACEMENT = (os.environ.get("CT_H3_IMAGE_PLACEMENT") or "auto").strip().lower()
WARMUP_MODE = (os.environ.get("CT_H3_IMAGE_WARMUP") or "server").strip().lower()
SGLANG_PORT = int(os.environ.get("CT_H3_SGLANG_PORT") or "30000")
SGLANG_BASE_WHEEL = "0.5.23"
SGLANG_MAIN_COMMIT = "b174f4bb4fe0f316cf625aa5441f0ea5bd65cd84"

H3_WORK = Path("/content/minimax-h3-image")
VENV_DIR = H3_WORK / "venv"
SGLANG_SRC_DIR = H3_WORK / "sglang-src"
MODEL_DIR = H3_WORK / "MiniMax-H3-Image"
SGLANG_OUTPUT_DIR = H3_WORK / "sglang-outputs"
SGLANG_INPUT_DIR = H3_WORK / "sglang-inputs"
UPLOADS_DIR = H3_WORK / "uploads"
JOBS_DIR = H3_WORK / "jobs"
CACHE_LOCAL_ROOT = H3_WORK / "compiler-caches"
FLASHINFER_CACHE_DIR = CACHE_LOCAL_ROOT / "flashinfer-cache"
TRITON_CACHE_DIR = CACHE_LOCAL_ROOT / "triton-cache"
INDUCTOR_CACHE_DIR = CACHE_LOCAL_ROOT / "inductor-cache"
SGLANG_JIT_CACHE_DIR = CACHE_LOCAL_ROOT / "sglang-jit-cache"

for _d in (
    H3_WORK, SGLANG_OUTPUT_DIR, SGLANG_INPUT_DIR, UPLOADS_DIR, JOBS_DIR,
    FLASHINFER_CACHE_DIR, TRITON_CACHE_DIR, INDUCTOR_CACHE_DIR, SGLANG_JIT_CACHE_DIR,
    Path(DATA_DIR) / "inputs", Path(DATA_DIR) / "outputs",
):
    _d.mkdir(parents=True, exist_ok=True)

gpu_info = ct_gpu_stat("name,memory.total,compute_cap")
if gpu_info == "cpu/no-gpu":
    raise RuntimeError("MiniMax-H3-Image requires a CUDA GPU (Colab G4 96 GB recommended).")
_gpu_parts = [p.strip() for p in gpu_info.splitlines()[0].split(",")]
GPU_NAME = _gpu_parts[0]
GPU_VRAM_MIB = int(re.sub(r"[^0-9]", "", _gpu_parts[1]) or "0") if len(_gpu_parts) > 1 else 0
GPU_CC = _gpu_parts[2] if len(_gpu_parts) > 2 else "unknown"
FREE_SSD_GB = shutil.disk_usage("/content").free / (1024**3)
h3log(f"GPU: {GPU_NAME} ({GPU_VRAM_MIB} MiB, sm={GPU_CC}) | /content free SSD: {FREE_SSD_GB:.1f} GiB")
h3log(f"Target config: model={MODEL_REPO} quant={QUANTIZATION} placement={PLACEMENT} warmup={WARMUP_MODE}")

(H3_WORK / "colab_cache.py").write_text('"""Stage optional compiler caches using one Drive archive per environment/GPU."""\nimport hashlib\nimport os\nimport shutil\nimport tarfile\nimport tempfile\nimport threading\nimport time\nimport uuid\nfrom pathlib import Path\n\n\ndef _progress(log, message, operation):\n    stopped = threading.Event()\n    started = time.monotonic()\n\n    def heartbeat():\n        while not stopped.wait(10):\n            log(f\'{message}，已用时 {time.monotonic()-started:.0f} 秒\')\n\n    log(message)\n    thread = threading.Thread(target=heartbeat, daemon=True)\n    thread.start()\n    try:\n        return operation()\n    finally:\n        stopped.set()\n        thread.join()\n\n\ndef _files(root):\n    return sorted(p for p in root.rglob(\'*\') if p.is_file() and not p.is_symlink()\n                  and not p.name.endswith((\'.lock\', \'.tmp\')))\n\n\ndef _fingerprint(root, files):\n    digest = hashlib.sha256()\n    for path in files:\n        stat = path.stat()\n        digest.update(f\'{path.relative_to(root)}:{stat.st_size}:{stat.st_mtime_ns}\\n\'.encode())\n    return digest.hexdigest()\n\n\ndef _unpack(archive, local):\n    # Validate the whole archive before extracting; compiler caches contain\n    # regular files/directories only. Never follow archive or local symlinks.\n    with tarfile.open(archive, \'r:\') as tar:\n        members = tar.getmembers()\n        root = local.resolve()\n        for member in members:\n            destination = local / member.name\n            if (not (member.isfile() or member.isdir()) or Path(member.name).is_absolute()\n                    or not destination.resolve().is_relative_to(root)):\n                raise ValueError(\'缓存归档包含不安全的路径或文件类型\')\n            current = destination\n            while current != local:\n                if current.is_symlink():\n                    raise ValueError(\'缓存目标包含符号链接\')\n                current = current.parent\n        tar.extractall(local, members=members, filter=\'data\')\n\n\ndef restore_cache(local, legacy, archive_dir, key, log=print):\n    local, legacy, archive_dir = map(Path, (local, legacy, archive_dir))\n    local.mkdir(parents=True, exist_ok=True)\n    archive = archive_dir / key / (local.name + \'.tar\')\n    stamp = local.parent / (\'.\' + local.name + \'-\' + key + \'.restored\')\n    if archive.is_file():\n        stat = archive.stat()\n        signature = f\'{stat.st_size}:{stat.st_mtime_ns}\'\n        if stamp.exists() and stamp.read_text() == signature and _files(local):\n            log(f\'{local.name}: 已在本地恢复，跳过重复传输\')\n            return \'local\'\n        try:\n            with tempfile.TemporaryDirectory(dir=local.parent, prefix=\'.cache-restore-\') as tmp:\n                staged = Path(tmp) / \'cache.tar\'\n                _progress(log, f\'{local.name}: 从网盘读取归档（{stat.st_size/1e6:.1f} MB）\',\n                          lambda: shutil.copyfile(archive, staged))\n                _progress(log, f\'{local.name}: 在本地 SSD 解包\', lambda: _unpack(staged, local))\n            stamp.write_text(signature)\n            (local.parent / (\'.\' + local.name + \'-\' + key + \'.saved\')).write_text(\n                _fingerprint(local, _files(local)))\n            log(f\'{local.name}: 归档恢复完成\')\n            return \'archive\'\n        except (OSError, ValueError, tarfile.TarError) as error:\n            log(f\'{local.name}: 归档不可用，回退旧缓存（{error}）\')\n    if legacy.is_dir():\n        try:\n            def migrate():\n                shutil.copytree(legacy, local, dirs_exist_ok=True, symlinks=True)\n            _progress(log, f\'{local.name}: 首次迁移旧版网盘缓存（小文件较多时耗时较长）\', migrate)\n            if _files(local):\n                save_cache(local, archive_dir, key, log)\n                return \'legacy\'\n        except (OSError, ValueError, tarfile.TarError) as error:\n            log(f\'{local.name}: 旧缓存迁移失败，回退按需生成（{error}）\')\n    log(f\'{local.name}: 无持久缓存，将由服务按需生成\')\n    return \'miss\'\n\n\ndef save_cache(local, archive_dir, key, log=print):\n    local, archive_dir = Path(local), Path(archive_dir)\n    files = _files(local)\n    if not files:\n        log(f\'{local.name}: 无缓存文件，跳过归档\')\n        return False\n    fingerprint = _fingerprint(local, files)\n    saved = local.parent / (\'.\' + local.name + \'-\' + key + \'.saved\')\n    destination = archive_dir / key / (local.name + \'.tar\')\n    if saved.exists() and saved.read_text() == fingerprint and destination.is_file():\n        log(f\'{local.name}: 内容未变化，跳过上传\')\n        return False\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    temporary = destination.with_name(destination.name + \'.\' + uuid.uuid4().hex + \'.tmp\')\n    try:\n        with tempfile.TemporaryDirectory(dir=local.parent, prefix=\'.cache-save-\') as tmp:\n            staged = Path(tmp) / \'cache.tar\'\n            def pack():\n                with tarfile.open(staged, \'w:\') as tar:\n                    for path in files:\n                        tar.add(path, arcname=str(path.relative_to(local)), recursive=False)\n            _progress(log, f\'{local.name}: 本地打包 {len(files)} 个文件\', pack)\n            _progress(log, f\'{local.name}: 保存归档到网盘（{staged.stat().st_size/1e6:.1f} MB）\',\n                      lambda: shutil.copyfile(staged, temporary))\n            os.replace(temporary, destination)\n        saved.write_text(fingerprint)\n        stat = destination.stat()\n        (local.parent / (\'.\' + local.name + \'-\' + key + \'.restored\')).write_text(\n            f\'{stat.st_size}:{stat.st_mtime_ns}\')\n        log(f\'{local.name}: 网盘归档已更新\')\n        return True\n    finally:\n        temporary.unlink(missing_ok=True)\n', encoding="utf-8")
sys.path.insert(0, str(H3_WORK))
from colab_cache import restore_cache, save_cache  # noqa: E402

CACHE_ARCHIVES = Path(DATA_DIR) / "cache-archives"
CACHE_KEY = re.sub(r"[^a-zA-Z0-9._-]+", "-", f"sglang-{SGLANG_MAIN_COMMIT[:10]}-{GPU_NAME}-{GPU_CC}").strip("-").lower()
CACHE_PAIRS = [
    (FLASHINFER_CACHE_DIR, Path(DATA_DIR) / "flashinfer-cache"),
    (TRITON_CACHE_DIR, Path(DATA_DIR) / "triton-cache"),
    (INDUCTOR_CACHE_DIR, Path(DATA_DIR) / "inductor-cache"),
    (SGLANG_JIT_CACHE_DIR, Path(DATA_DIR) / "sglang-jit-cache"),
]
for _loc_c, _leg_c in CACHE_PAIRS:
    restore_cache(_loc_c, _leg_c, CACHE_ARCHIVES, CACHE_KEY, h3log)

def run_logged(cmd, log_path, *, env=None, cwd=None):
    started = time.monotonic()
    with open(log_path, "a", encoding="utf-8") as lf:
        lf.write(f"\n$ {' '.join(map(str, cmd))}\n")
        lf.flush()
        proc = subprocess.Popen(cmd, stdout=lf, stderr=subprocess.STDOUT, env=env, cwd=cwd)
        last_beat = started
        while proc.poll() is None:
            time.sleep(2)
            if time.monotonic() - last_beat >= 20:
                h3log(f"Running {Path(cmd[0]).name} ({int(time.monotonic() - started)}s elapsed)...")
                last_beat = time.monotonic()
        if proc.returncode != 0:
            tail = "".join(open(log_path, errors="replace").readlines()[-45:])
            raise RuntimeError(f"Command failed (exit {proc.returncode}): {' '.join(map(str, cmd))}\n{tail}")

def prepare_env():
    py_bin = VENV_DIR / "bin" / "python"
    marker = VENV_DIR / ".h3_env_ready"
    stamp_val = f"{SGLANG_BASE_WHEEL}:{SGLANG_MAIN_COMMIT}:v2"
    if py_bin.is_file() and marker.is_file() and marker.read_text(encoding="utf-8").strip() == stamp_val:
        chk = subprocess.run(
            [str(py_bin), "-c",
             "from sglang.multimodal_gen.configs.pipeline_configs.minimax_h3_image import MiniMaxH3ImagePipelineConfig; "
             "import blake3, uvicorn, torch; assert torch.cuda.is_available()"],
            check=False, capture_output=True, text=True,
        )
        if chk.returncode == 0:
            h3log("Reusing prepared SGLang Diffusion virtualenv on local SSD")
            return py_bin

    env_log = f"{WORK}/h3_setup.log"
    if not shutil.which("uv"):
        run_logged([sys.executable, "-m", "pip", "install", "-q", "uv"], env_log)
    uv_bin = shutil.which("uv") or "uv"

    h3log(f"Creating clean Python 3.12 venv at {VENV_DIR} (log: {env_log})...")
    run_logged([uv_bin, "venv", "--python", "3.12", str(VENV_DIR), "--clear"], env_log)

    h3log(f"Installing prebuilt SGLang {SGLANG_BASE_WHEEL} + CUDA 13 + FlashAttention-4 base wheels...")
    run_logged(
        [
            uv_bin, "pip", "install",
            "--python", str(py_bin),
            "--prerelease=allow",
            f"sglang[diffusion]=={SGLANG_BASE_WHEEL}",
            "flash-attn-4>=4.0.0b18",
            "blake3>=1.0.0",
            "huggingface_hub>=0.27.0",
            "uvicorn>=0.30.0",
            "setuptools>=68.0",
            "setuptools-scm>=8.0",
            "wheel",
        ],
        env_log,
    )

    h3log(f"Fetching SGLang commit {SGLANG_MAIN_COMMIT[:10]} (MiniMax-H3-Image native pipeline)...")
    if not (SGLANG_SRC_DIR / ".git").is_dir():
        shutil.rmtree(SGLANG_SRC_DIR, ignore_errors=True)
        SGLANG_SRC_DIR.mkdir(parents=True, exist_ok=True)
        run_logged(["git", "-C", str(SGLANG_SRC_DIR), "init"], env_log)
        run_logged(["git", "-C", str(SGLANG_SRC_DIR), "remote", "add", "origin", "https://github.com/sgl-project/sglang.git"], env_log)
    run_logged(["git", "-C", str(SGLANG_SRC_DIR), "fetch", "--depth", "1", "origin", SGLANG_MAIN_COMMIT], env_log)
    run_logged(["git", "-C", str(SGLANG_SRC_DIR), "checkout", "-f", "FETCH_HEAD"], env_log)

    # Preserve prebuilt Rust .so extensions from the PyPI binary wheel before overlaying pure-Python main!
    wheel_pkg = Path(
        subprocess.check_output(
            [str(py_bin), "-c", "import pathlib, sglang; print(pathlib.Path(sglang.__file__).resolve().parent)"],
            text=True,
        ).strip()
    )
    src_pkg = SGLANG_SRC_DIR / "python" / "sglang"
    copied_so = 0
    if wheel_pkg.is_dir() and wheel_pkg != src_pkg.resolve():
        for so_file in wheel_pkg.rglob("*.so"):
            rel = so_file.relative_to(wheel_pkg)
            dst = src_pkg / rel
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(so_file, dst)
            copied_so += 1
    h3log(f"Preserved {copied_so} prebuilt native extension(s) into {src_pkg}")

    overlay_env = dict(os.environ)
    overlay_env["SGLANG_BUILD_RUST_EXTS"] = "none"
    overlay_env["SETUPTOOLS_SCM_PRETEND_VERSION"] = "0.5.24.dev0"
    run_logged(
        [
            uv_bin, "pip", "install",
            "--python", str(py_bin),
            "--no-build-isolation",
            "--no-deps",
            "-e", str(SGLANG_SRC_DIR / "python"),
        ],
        env_log,
        env=overlay_env,
    )

    h3log("Aligning CUDA 13 kernel packages with SGLang main...")
    run_logged(
        [
            uv_bin, "pip", "install",
            "--python", str(py_bin),
            "--prerelease=allow",
            "blake3>=1.0.0",
            "flashinfer_python[cu13]==0.7.3",
            "humming-kernels[cu13]==0.1.18",
            "sgl-deep-gemm==0.2.7",
            "sgl-flash-mla==1.0.4",
            "pillow<13",
        ],
        env_log,
    )
    shutil.rmtree("/root/.cache/uv", ignore_errors=True)

    run_logged(
        [
            str(py_bin), "-c",
            "import torch, uvicorn\n"
            "from sglang.multimodal_gen.configs.pipeline_configs.minimax_h3_image import MiniMaxH3ImagePipelineConfig\n"
            "from sglang.multimodal_gen.runtime.pipelines.minimax_h3_pipeline import MiniMaxH3ImagePipeline\n"
            "print('sglang-h3-image-ok', torch.__version__, 'cuda=', torch.cuda.is_available())",
        ],
        env_log,
    )
    marker.write_text(stamp_val + "\n", encoding="utf-8")
    h3log("SGLang Diffusion environment ready")
    return py_bin

def _patch_local_model_index(model_dir: Path):
    """Point external text_encoder spec to the already-downloaded local subfolder to save 66.7 GB SSD."""
    for fname in ("model_index.json", "modular_model_index.json"):
        idx_path = model_dir / fname
        if not idx_path.is_file():
            continue
        data = json.loads(idx_path.read_text(encoding="utf-8"))
        changed = False
        for k, spec in list(data.items()):
            if isinstance(spec, list) and len(spec) == 3 and isinstance(spec[2], dict):
                sub = spec[2].get("subfolder")
                if isinstance(sub, str) and sub and (model_dir / sub / "config.json").is_file():
                    if spec[2].get("pretrained_model_name_or_path") != str(model_dir):
                        spec[2]["pretrained_model_name_or_path"] = str(model_dir)
                        changed = True
            elif isinstance(spec, dict) and isinstance(spec.get("subfolder"), str):
                sub = spec["subfolder"]
                if (model_dir / sub / "config.json").is_file() and spec.get("pretrained_model_name_or_path") != str(model_dir):
                    spec["pretrained_model_name_or_path"] = str(model_dir)
                    changed = True
        if changed:
            tmp = idx_path.with_name(f".{fname}.tmp")
            tmp.write_text(json.dumps(data, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
            os.replace(tmp, idx_path)
            h3log(f"Patched {fname} -> local text_encoder at {model_dir / 'text_encoder'}")

def stage_model(py_bin: Path):
    required = (
        MODEL_DIR / "model_index.json",
        MODEL_DIR / "pdd_config.json",
        MODEL_DIR / "pdd_fused_heads.safetensors",
        MODEL_DIR / "transformer" / "config.json",
        MODEL_DIR / "text_encoder" / "config.json",
        MODEL_DIR / "vae" / "config.json",
    )
    marker = MODEL_DIR / ".h3_model_complete"
    if all(p.is_file() for p in required) and marker.is_file():
        _patch_local_model_index(MODEL_DIR)
        h3log(f"Reusing staged {MODEL_REPO} weights at {MODEL_DIR}")
        return

    free_gb = shutil.disk_usage("/content").free / (1024**3)
    if free_gb < 142.0 and not (MODEL_DIR / "transformer").is_dir():
        raise RuntimeError(
            f"Insufficient local SSD space on /content ({free_gb:.1f} GiB free; need >= 142 GiB for 135.3 GB MiniMax-H3-Image)."
        )

    h3log(f"Downloading {MODEL_REPO} (skipping duplicate *_diffusers copies; ~135.3 GB) -> {MODEL_DIR} ...")
    dl_env = dict(os.environ)
    dl_env.update({
        "HF_XET_CHUNK_CACHE_SIZE_BYTES": "0",
        "HF_HUB_ENABLE_HF_TRANSFER": "0",
        "HF_HUB_DISABLE_TELEMETRY": "1",
    })
    dl_log = f"{WORK}/h3_download.log"
    run_logged(
        [
            str(py_bin), "-c",
            "from huggingface_hub import snapshot_download\n"
            f"snapshot_download(\n"
            f"    repo_id={MODEL_REPO!r},\n"
            f"    local_dir={str(MODEL_DIR)!r},\n"
            "    ignore_patterns=[\n"
            "        'transformer_diffusers/*', 'vae_diffusers/*',\n"
            "        'assets/*', 'examples/*', '*.md', '.gitattributes',\n"
            "    ],\n"
            "    max_workers=8,\n"
            ")\n",
        ],
        dl_log,
        env=dl_env,
    )
    shutil.rmtree(MODEL_DIR / ".cache", ignore_errors=True)
    for _hfc in ("/root/.cache/huggingface/xet", "/root/.cache/huggingface/hub"):
        shutil.rmtree(_hfc, ignore_errors=True)
    _patch_local_model_index(MODEL_DIR)
    marker.write_text(f"{MODEL_REPO}\n", encoding="utf-8")
    h3log(f"Model staged cleanly; /content free SSD remaining: {shutil.disk_usage('/content').free / (1024**3):.1f} GiB")

PY_BIN = prepare_env()
stage_model(PY_BIN)
H3_SETUP_OK = True
h3log("Setup complete")


In [ ]:
# @title 2. Service -- launch `sglang serve` on loopback + authenticated MiniMax-H3-Image gateway
from pathlib import Path
if not globals().get("H3_SETUP_OK"):
    raise RuntimeError("MiniMax-H3-Image setup cell failed before completing; check setup traceback above.")

(H3_WORK / "colab_asgi_common.py").write_text('"""Shared security, Drive path containment, and audio helpers for Colab ASGI middlewares."""\nfrom __future__ import annotations\n\nimport base64\nimport glob as glob_mod\nimport os\nimport re\nimport secrets\nimport subprocess\nimport time\nfrom pathlib import Path\nfrom typing import Any\n\nAUDIO_EXTS = (\n    ".wav", ".flac", ".mp3", ".ogg", ".m4a", ".aac", ".opus", ".webm", ".mp4", ".mkv", ".wma", ".mpga", ".mpeg",\n)\nDEFAULT_SKIP_DIRS = {\n    ".git", "node_modules", "__pycache__", ".venv", "venv", "aux-venv", "custom_nodes",\n    "models", ".runtimes", "Takeout", ".Trash", ".ipynb_checkpoints",\n    "flashinfer-cache", "triton-cache", "vllm-cache", "cache-archives", "kernel-archives",\n}\nVOICE_NAME_RE = re.compile(r"^[a-z0-9][a-z0-9._-]{0,63}$")\n\n\ndef validate_voice_name(name: str) -> str:\n    """Validate and normalize a custom voice identifier against path traversal."""\n    vname = (name or "").strip().lower()\n    if not vname or ".." in vname or not VOICE_NAME_RE.fullmatch(vname):\n        raise ValueError(f"Invalid voice name \'{name}\': use 1-64 lowercase [a-z0-9._-] characters without \'..\'")\n    return vname\n\n\ndef _get_header(headers: Any, key: str) -> str:\n    val = headers.get(key)\n    if val:\n        return str(val)\n    if isinstance(headers, dict):\n        low = key.lower()\n        for k, v in headers.items():\n            if str(k).lower() == low and v:\n                return str(v)\n    return ""\n\n\ndef verify_bearer_auth(headers: Any, api_token: str) -> bool:\n    """Fail-closed constant-time Bearer / X-API-Key authentication check."""\n    if not api_token:\n        return False\n    expected_bearer = f"Bearer {api_token}"\n    auth = _get_header(headers, "authorization")\n    if auth and secrets.compare_digest(auth.encode("utf-8"), expected_bearer.encode("utf-8")):\n        return True\n    xkey = _get_header(headers, "x-api-key")\n    if xkey and secrets.compare_digest(xkey.encode("utf-8"), api_token.encode("utf-8")):\n        return True\n    return False\n\n\ndef sanitize_header_param(value: str) -> str:\n    """Strip CR/LF/NUL and escape quotes/backslashes for Content-Disposition parameters."""\n    cleaned = (value or "").replace("\\r", "").replace("\\n", "").replace("\\x00", "")\n    return cleaned.replace("\\\\", "\\\\\\\\").replace(\'"\', \'\\\\"\')\n\n\ndef build_multipart_form(\n    fields: list[tuple[str, str]],\n    files: list[tuple[str, str, str, bytes]] | None = None,\n) -> tuple[str, bytes]:\n    """Build a safe multipart/form-data body returning (content_type, body_bytes)."""\n    boundary = f"----ctform{secrets.token_hex(16)}"\n    parts: list[bytes] = []\n    for key, val in fields:\n        safe_key = sanitize_header_param(str(key))\n        parts.append(\n            f\'--{boundary}\\r\\nContent-Disposition: form-data; name="{safe_key}"\\r\\n\\r\\n{val}\\r\\n\'.encode("utf-8")\n        )\n    for field_name, filename, content_type, data in files or []:\n        safe_field = sanitize_header_param(str(field_name))\n        safe_filename = sanitize_header_param(str(filename))\n        safe_ct = (content_type or "application/octet-stream").replace("\\r", "").replace("\\n", "").replace("\\x00", "")\n        parts.append(\n            (\n                f"--{boundary}\\r\\n"\n                f\'Content-Disposition: form-data; name="{safe_field}"; filename="{safe_filename}"\\r\\n\'\n                f"Content-Type: {safe_ct}\\r\\n\\r\\n"\n            ).encode("utf-8")\n            + data\n            + b"\\r\\n"\n        )\n    parts.append(f"--{boundary}--\\r\\n".encode("utf-8"))\n    return f"multipart/form-data; boundary={boundary}", b"".join(parts)\n\n\ndef allowed_drive_roots(\n    data_dir: Path,\n    mydrive_root: Path = Path("/content/drive/MyDrive"),\n) -> tuple[Path, ...]:\n    """Return resolved root directories that Drive/data operations may access."""\n    roots: list[Path] = []\n    if mydrive_root.is_dir():\n        roots.append(mydrive_root.resolve())\n    roots.append(data_dir.resolve())\n    return tuple(dict.fromkeys(roots))\n\n\ndef _ensure_within_roots(candidate: Path, roots: tuple[Path, ...], raw_spec: str) -> Path:\n    resolved = candidate.resolve()\n    if not any(resolved.is_relative_to(root) for root in roots):\n        raise ValueError(f"Path \'{raw_spec}\' escapes allowed Drive/data directory")\n    return resolved\n\n\ndef resolve_drive_path(\n    raw_spec: str,\n    *,\n    data_dir: Path,\n    default_subdir: str | None = None,\n    mydrive_root: Path = Path("/content/drive/MyDrive"),\n    drive_root: Path = Path("/content/drive"),\n) -> Path:\n    """Resolve a Drive or service-data path strictly inside allowed roots."""\n    if "\\x00" in (raw_spec or ""):\n        raise ValueError("Invalid NUL byte in path")\n    roots = allowed_drive_roots(data_dir, mydrive_root)\n    spec = (raw_spec or "").strip()\n\n    is_file_scheme = spec.startswith("file://")\n    for prefix in ("drive://", "gdrive://", "file://"):\n        if spec.startswith(prefix):\n            spec = spec[len(prefix):]\n            break\n\n    if is_file_scheme:\n        candidate = Path("/" + spec.lstrip("/"))\n        return _ensure_within_roots(candidate, roots, raw_spec)\n\n    had_leading_slash = spec.startswith("/")\n    clean = spec.lstrip("/")\n    if not clean:\n        if default_subdir and (data_dir / default_subdir).is_dir():\n            return _ensure_within_roots(data_dir / default_subdir, roots, raw_spec)\n        return roots[0]\n\n    if had_leading_slash and not clean.startswith(("content/drive/", "drive/", "MyDrive/", "MyDrive")):\n        # Allow explicit absolute path ONLY if it already lies inside an allowed root (e.g. DATA_DIR in --no-drive mode)\n        return _ensure_within_roots(Path("/" + clean), roots, raw_spec)\n\n    candidates: list[Path] = []\n    if clean.startswith("content/drive/"):\n        candidates.append(Path("/") / clean)\n    elif clean.startswith("drive/"):\n        candidates.append(drive_root.parent / clean)\n    elif clean.startswith("MyDrive/"):\n        candidates.append(drive_root / clean)\n    elif clean == "MyDrive":\n        candidates.append(mydrive_root)\n    elif clean.startswith("colab-llm/") and mydrive_root.is_dir():\n        candidates.append(mydrive_root / clean)\n        candidates.append(data_dir / clean)\n    else:\n        first_seg = clean.split("/", 1)[0]\n        already_prefixed = bool(\n            default_subdir\n            and (\n                clean == default_subdir\n                or clean.startswith(f"{default_subdir}/")\n                or (first_seg != clean and (data_dir / first_seg).is_dir())\n            )\n        )\n        if default_subdir and not already_prefixed:\n            candidates.append(data_dir / default_subdir / clean)\n        candidates.append(data_dir / clean)\n        if default_subdir and already_prefixed:\n            candidates.append(data_dir / default_subdir / clean)\n        if mydrive_root.is_dir():\n            candidates.append(mydrive_root / clean)\n\n    checked: list[Path] = []\n    for cand in candidates:\n        resolved = _ensure_within_roots(cand, roots, raw_spec)\n        checked.append(resolved)\n        if resolved.exists():\n            return resolved\n    if not checked:\n        raise ValueError(f"Path \'{raw_spec}\' could not be resolved inside allowed Drive/data directory")\n    return checked[0]\n\n\ndef rel_to_mydrive(p: Path, mydrive_root: Path = Path("/content/drive/MyDrive")) -> str:\n    try:\n        return str(p.resolve().relative_to(mydrive_root.resolve()))\n    except Exception:\n        return str(p)\n\n\ndef list_drive_audio_files(\n    spec: str,\n    *,\n    data_dir: Path,\n    default_subdir: str | None = None,\n    recursive: bool = True,\n    mydrive_root: Path = Path("/content/drive/MyDrive"),\n    drive_root: Path = Path("/content/drive"),\n    skip_dirs: set[str] | None = None,\n) -> list[dict[str, Any]]:\n    """List audio files matching a Drive path or glob strictly inside allowed roots."""\n    if "\\x00" in (spec or ""):\n        raise ValueError("Invalid NUL byte in path")\n    roots = allowed_drive_roots(data_dir, mydrive_root)\n    ignored = skip_dirs if skip_dirs is not None else DEFAULT_SKIP_DIRS\n    raw = (spec or "").strip()\n    is_file_scheme = raw.startswith("file://")\n    for prefix in ("drive://", "gdrive://", "file://"):\n        if raw.startswith(prefix):\n            raw = raw[len(prefix):]\n            break\n\n    has_glob = any(ch in raw for ch in ("*", "?", "["))\n    matched_paths: list[Path] = []\n\n    if has_glob:\n        if is_file_scheme:\n            bases = [Path("/" + raw.lstrip("/"))]\n        else:\n            clean = raw.lstrip("/")\n            if raw.startswith("/") and not clean.startswith(("content/drive/", "drive/", "MyDrive/")):\n                bases = [Path("/" + clean)]\n            elif clean.startswith("content/drive/"):\n                bases = [Path("/") / clean]\n            elif clean.startswith("MyDrive/"):\n                bases = [drive_root / clean]\n            else:\n                bases = []\n                if default_subdir:\n                    bases.append(data_dir / default_subdir / clean)\n                bases.append(data_dir / clean)\n                if mydrive_root.is_dir():\n                    bases.append(mydrive_root / clean)\n\n        # Ensure static prefix before wildcard does not escape allowed roots\n        for pattern_path in bases:\n            prefix_parts: list[str] = []\n            for part in pattern_path.parts:\n                if any(ch in part for ch in ("*", "?", "[")):\n                    break\n                prefix_parts.append(part)\n            if prefix_parts:\n                _ensure_within_roots(Path(*prefix_parts), roots, spec)\n\n        seen: set[str] = set()\n        for pattern_path in bases:\n            for hit in sorted(glob_mod.glob(str(pattern_path), recursive=True)):\n                hp = Path(hit).resolve()\n                if not any(hp.is_relative_to(r) for r in roots):\n                    continue\n                if hp.is_file() and hp.suffix.lower() in AUDIO_EXTS and str(hp) not in seen:\n                    seen.add(str(hp))\n                    matched_paths.append(hp)\n    else:\n        target = resolve_drive_path(\n            spec,\n            data_dir=data_dir,\n            default_subdir=default_subdir,\n            mydrive_root=mydrive_root,\n            drive_root=drive_root,\n        )\n        if target.is_file():\n            if target.suffix.lower() in AUDIO_EXTS:\n                matched_paths.append(target)\n        elif target.is_dir():\n            if not recursive:\n                for hp in sorted(target.iterdir()):\n                    resolved = hp.resolve()\n                    if any(resolved.is_relative_to(r) for r in roots) and resolved.is_file() and resolved.suffix.lower() in AUDIO_EXTS:\n                        matched_paths.append(resolved)\n            else:\n                for root_dir, dirnames, filenames in os.walk(target):\n                    dirnames[:] = sorted(d for d in dirnames if d not in ignored and not d.startswith("."))\n                    for fn in sorted(filenames):\n                        hp = (Path(root_dir) / fn).resolve()\n                        if any(hp.is_relative_to(r) for r in roots) and hp.is_file() and hp.suffix.lower() in AUDIO_EXTS:\n                            matched_paths.append(hp)\n\n    items: list[dict[str, Any]] = []\n    for p in matched_paths:\n        st = p.stat()\n        items.append({\n            "drive_path": str(p),\n            "rel_path": rel_to_mydrive(p, mydrive_root),\n            "name": p.name,\n            "size_bytes": st.st_size,\n            "size_mb": round(st.st_size / 1e6, 2),\n            "mtime": int(st.st_mtime),\n        })\n    return items\n\n\ndef audio_path_to_wav_data_url(\n    audio_path: Path,\n    max_duration_sec: float = 24.0,\n    sample_rate: int = 24000,\n    timeout: int = 60,\n) -> str:\n    """Decode a Drive audio file to a mono PCM16 WAV data URL with bounded ffmpeg timeout."""\n    if not audio_path.is_file():\n        raise FileNotFoundError(f"Audio file not found: {audio_path}")\n    cmd = [\n        "ffmpeg", "-nostdin", "-v", "error",\n        "-t", f"{max_duration_sec:.1f}",\n        "-i", str(audio_path),\n        "-vn", "-ac", "1", "-ar", str(sample_rate), "-c:a", "pcm_s16le", "-f", "wav", "pipe:1",\n    ]\n    try:\n        res = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, timeout=timeout, check=False)\n    except OSError as os_err:\n        if audio_path.suffix.lower() == ".wav":\n            b64 = base64.b64encode(audio_path.read_bytes()).decode("ascii")\n            return f"data:audio/wav;base64,{b64}"\n        raise RuntimeError(f"ffmpeg failed decoding {audio_path.name}: {os_err}") from os_err\n    if res.returncode != 0 or not res.stdout:\n        if audio_path.suffix.lower() == ".wav":\n            b64 = base64.b64encode(audio_path.read_bytes()).decode("ascii")\n            return f"data:audio/wav;base64,{b64}"\n        err = res.stderr.decode("utf-8", errors="replace").strip()\n        raise RuntimeError(f"ffmpeg failed decoding {audio_path.name}: {err}")\n    b64 = base64.b64encode(res.stdout).decode("ascii")\n    return f"data:audio/wav;base64,{b64}"\n\n\ndef save_output_audio_to_drive(\n    audio_bytes: bytes,\n    spec: str,\n    *,\n    data_dir: Path,\n    output_dir: Path,\n    prefix: str,\n    fmt: str = "wav",\n    mydrive_root: Path = Path("/content/drive/MyDrive"),\n    drive_root: Path = Path("/content/drive"),\n) -> str:\n    """Persist synthesized audio bytes safely inside Drive or service output directory."""\n    output_dir.mkdir(parents=True, exist_ok=True)\n    ext = "." + re.sub(r"[^a-z0-9]", "", (fmt or "wav").lower())\n    if not ext or ext == ".":\n        ext = ".wav"\n    if spec and spec not in ("auto", "true", "True", "1"):\n        dest = resolve_drive_path(\n            spec,\n            data_dir=data_dir,\n            mydrive_root=mydrive_root,\n            drive_root=drive_root,\n        )\n        if dest.is_dir() or spec.endswith("/"):\n            dest.mkdir(parents=True, exist_ok=True)\n            dest = dest / f"{prefix}-{time.strftime(\'%Y%m%d-%H%M%S\')}{ext}"\n        else:\n            dest.parent.mkdir(parents=True, exist_ok=True)\n            if not dest.suffix:\n                dest = dest.with_suffix(ext)\n    else:\n        dest = output_dir / f"{prefix}-{time.strftime(\'%Y%m%d-%H%M%S\')}{ext}"\n    dest.write_bytes(audio_bytes)\n    return str(dest)\n\n\ndef tail_service_log(\n    log_file: Path = Path("/content/ct/service.log"),\n    lines: int = 120,\n) -> dict[str, Any]:\n    n_lines = max(1, min(int(lines), 2000))\n    entries = log_file.read_text(encoding="utf-8", errors="replace").splitlines()[-n_lines:] if log_file.is_file() else []\n    return {"lines": len(entries), "log": "\\n".join(entries)}\n', encoding="utf-8")
(H3_WORK / "minimax_h3_image_middleware.py").write_text('"""Authenticated ASGI gateway, async job queue, binary upload/download & Drive I/O for MiniMax-H3-Image.\n\nSits on `127.0.0.1:<CT_SERVICE_PORT>` (default `8092`) in front of `sglang serve` on loopback\n`127.0.0.1:<CT_H3_SGLANG_PORT>` (default `30000`).\n\nNetwork-friendly design:\n  1. Async job submission & tiny JSON polling (`POST /api/ct/jobs`, `GET /api/ct/jobs/<id>`) so CLI\n     calls never hold long idle HTTP connections over Cloudflare Tunnel / FRP / proxies.\n  2. Content-addressed raw binary reference upload (`HEAD/GET /api/ct/uploads/<sha256>` +\n     `POST /api/ct/uploads`) avoiding +33% Base64 bloat and deduplicating repeated edits to 0 bytes.\n  3. Zero-Base64 loopback file handoff with SGLang (`response_format="url"` + `output_compression=0`\n     into `/content/minimax-h3-image/sglang-outputs`), plus server-side PNG compression or WebP/JPEG\n     transcoding and resumable `Range` binary streaming (`GET /api/ct/jobs/<id>/content`).\n  4. Zero-upload / zero-download Google Drive I/O (`drive://...` inputs & `save_to_drive` outputs).\n  5. Full OpenAI `/v1/images/generations`, `/v1/images/edits`, `/v1/models`, and `/health` compatibility,\n     all protected fail-closed by `verify_bearer_auth`.\n\nImports cleanly with only the Python standard library so local unit tests can exercise all security,\nnormalization, job queue, upload deduplication, and binary range-streaming paths without GPU deps.\n"""\nfrom __future__ import annotations\n\nimport asyncio\nimport base64\nfrom concurrent.futures import ThreadPoolExecutor\nfrom email.parser import BytesParser\nfrom email.policy import default as email_default_policy\nimport hashlib\nimport io\nimport json\nimport math\nimport mimetypes\nimport os\nfrom pathlib import Path\nimport re\nimport secrets\nimport shutil\nimport struct\nimport time\nfrom typing import Any, Callable\nimport urllib.error\nimport urllib.parse\nimport urllib.request\n\nfrom colab_asgi_common import (\n    allowed_drive_roots,\n    build_multipart_form,\n    rel_to_mydrive,\n    resolve_drive_path,\n    sanitize_header_param,\n    tail_service_log,\n    verify_bearer_auth,\n)\n\n\ndef _env_int(name: str, default: int) -> int:\n    raw = (os.environ.get(name) or "").strip()\n    if not raw:\n        return default\n    try:\n        return int(raw)\n    except ValueError:\n        return default\n\n\nSERVICE_NAME = os.environ.get("CT_SERVICE") or "minimax-h3-image"\nSERVICE_PORT = _env_int("CT_SERVICE_PORT", 8092)\nSGLANG_PORT = _env_int("CT_H3_SGLANG_PORT", 30000)\nSGLANG_BASE_URL = (os.environ.get("CT_H3_SGLANG_URL") or f"http://127.0.0.1:{SGLANG_PORT}").rstrip("/")\n\nAPI_TOKEN = os.environ.get("CT_API_TOKEN", "")\nMODEL_REPO = (os.environ.get("CT_H3_IMAGE_MODEL") or "MiniMaxAI/MiniMax-H3-Image").strip()\nMODEL_ALIAS = (os.environ.get("CT_H3_IMAGE_ALIAS") or "minimax-h3-image").strip()\nQUANTIZATION = (os.environ.get("CT_H3_IMAGE_QUANTIZATION") or "fp8-all").strip()\nPLACEMENT = (os.environ.get("CT_H3_IMAGE_PLACEMENT") or "auto").strip()\nWARMUP_MODE = (os.environ.get("CT_H3_IMAGE_WARMUP") or "server").strip()\n\nDRIVE_ROOT = Path("/content/drive")\nMYDRIVE_ROOT = DRIVE_ROOT / "MyDrive"\nDATA_DIR = Path(os.environ.get("CT_DATA_DIR") or f"/content/drive/MyDrive/colab-llm/{SERVICE_NAME}")\nDRIVE_INPUTS_DIR = DATA_DIR / "inputs"\nDRIVE_OUTPUTS_DIR = DATA_DIR / "outputs"\n\nLOCAL_WORK_DIR = Path(os.environ.get("CT_H3_LOCAL_WORK_DIR") or "/content/minimax-h3-image")\nSGLANG_OUTPUT_DIR = LOCAL_WORK_DIR / "sglang-outputs"\nSGLANG_INPUT_DIR = LOCAL_WORK_DIR / "sglang-inputs"\nUPLOADS_DIR = LOCAL_WORK_DIR / "uploads"\nJOBS_DIR = LOCAL_WORK_DIR / "jobs"\nLOG_FILE = Path("/content/ct/service.log")\nSGLANG_LOG_FILE = Path("/content/ct/sglang.log")\n\nDIM_ALIGNMENT = 32\nMIN_DIM = 64\nMAX_DIM = 8192\nFIXED_PDD_STEPS = 6\nDEFAULT_MIN_GENERATION_AREA = 2048 * 2048  # 4,194,304\nMIN_GENERATION_AREA_BOUNDS = (1024, 4096 * 4096)  # [1024, 16,777,216]\nMAX_REFERENCE_IMAGES = 16\nMAX_IMAGES_PER_JOB = 8\nMAX_JSON_BODY_BYTES = 96 * 1024 * 1024\nMAX_UPLOAD_BYTES = 80 * 1024 * 1024\nMAX_RETAINED_JOBS = 48\nMAX_UPLOADS_CACHE_BYTES = 4 * 1024 * 1024 * 1024  # 4 GiB\nSTREAM_CHUNK_BYTES = 65536\n\nIMAGE_EXTS = (".png", ".jpg", ".jpeg", ".webp", ".bmp", ".gif", ".tiff", ".tif")\nMIME_BY_FORMAT = {\n    "png": "image/png",\n    "webp": "image/webp",\n    "jpeg": "image/jpeg",\n    "jpg": "image/jpeg",\n}\nEXT_BY_FORMAT = {\n    "png": ".png",\n    "webp": ".webp",\n    "jpeg": ".jpg",\n    "jpg": ".jpg",\n}\n\n# Official MiniMax-H3-Image native 4K (16 MP) resolutions and aligned 2K (4 MP) / 1K (1 MP) counterparts.\nASPECT_PRESETS: dict[str, dict[str, tuple[int, int]]] = {\n    "4k": {\n        "1:1": (4096, 4096),\n        "4:3": (4736, 3552),\n        "3:4": (3552, 4736),\n        "3:2": (5024, 3360),\n        "2:3": (3360, 5024),\n        "16:9": (5472, 3072),\n        "9:16": (3072, 5472),\n        "21:9": (6272, 2688),\n        "9:21": (2688, 6272),\n    },\n    "2k": {\n        "1:1": (2048, 2048),\n        "4:3": (2368, 1760),\n        "3:4": (1760, 2368),\n        "3:2": (2496, 1664),\n        "2:3": (1664, 2496),\n        "16:9": (2752, 1536),\n        "9:16": (1536, 2752),\n        "21:9": (3136, 1344),\n        "9:21": (1344, 3136),\n    },\n    "1k": {\n        "1:1": (1024, 1024),\n        "4:3": (1184, 896),\n        "3:4": (896, 1184),\n        "3:2": (1248, 832),\n        "2:3": (832, 1248),\n        "16:9": (1376, 768),\n        "9:16": (768, 1376),\n        "21:9": (1568, 672),\n        "9:21": (672, 1568),\n    },\n}\nTIER_TARGET_AREAS = {\n    "4k": 4096 * 4096,\n    "2k": 2048 * 2048,\n    "1k": 1024 * 1024,\n}\n\nMODEL_ALIASES = {\n    "",\n    "minimax-h3-image",\n    "h3-image",\n    "minimaxai/minimax-h3-image",\n    "gpt-image-1",\n}\n\nUPLOAD_ID_RE = re.compile(r"^[0-9a-f]{64}$")\nJOB_ID_RE = re.compile(r"^job_[0-9a-f]{16,32}$")\nQWEN_IMAGE_TAG_RE = re.compile(r"<image\\s*(\\d+)>", re.IGNORECASE)\n\nCORS_HEADERS: list[tuple[bytes, bytes]] = [\n    (b"access-control-allow-origin", b"*"),\n    (b"access-control-allow-methods", b"GET, HEAD, POST, DELETE, OPTIONS"),\n    (b"access-control-allow-headers", b"Authorization, Content-Type, X-API-Key, X-Filename, Range"),\n    (\n        b"access-control-expose-headers",\n        b"Content-Length, Content-Range, Accept-Ranges, ETag, X-Job-Id, "\n        b"X-Image-Width, X-Image-Height, X-Image-Format, X-Saved-Drive-Path",\n    ),\n]\n\n\ndef _inject_cors_headers(headers: list[tuple[bytes, bytes]]) -> list[tuple[bytes, bytes]]:\n    existing = {k.lower() for k, _ in headers}\n    merged = list(headers)\n    for k, v in CORS_HEADERS:\n        if k.lower() not in existing:\n            merged.append((k, v))\n    return merged\n\n\ndef normalize_model_name(raw_model: str | None) -> str:\n    cleaned = (raw_model or "").strip()\n    if cleaned.lower() in MODEL_ALIASES or cleaned.lower() == MODEL_ALIAS.lower():\n        return MODEL_REPO\n    return cleaned\n\n\ndef normalize_edit_prompt_tags(prompt: str) -> str:\n    """Translate `<image1>`/`<image 1>` tags to MiniMax-H3-Image `<Picture 1>` when no `<Picture` tag is present."""\n    if not prompt or "<picture" in prompt.lower():\n        return prompt\n    return QWEN_IMAGE_TAG_RE.sub(lambda m: f"<Picture {int(m.group(1))}>", prompt)\n\n\ndef normalize_tier(raw_tier: str | None, *, default: str = "4k") -> str:\n    t = (raw_tier or default).strip().lower()\n    aliases = {\n        "4k": "4k",\n        "16mp": "4k",\n        "native": "4k",\n        "2k": "2k",\n        "4mp": "2k",\n        "1k": "1k",\n        "1mp": "1k",\n        "preview": "1k",\n    }\n    if t not in aliases:\n        raise ValueError(f"Unsupported resolution tier \'{raw_tier}\': expected 4k (16mp), 2k (4mp), or 1k (1mp)")\n    return aliases[t]\n\n\ndef align_dim(value: int) -> int:\n    aligned = int(round(value / DIM_ALIGNMENT)) * DIM_ALIGNMENT\n    return max(MIN_DIM, min(MAX_DIM, aligned))\n\n\ndef resolve_dimensions(\n    *,\n    size: str | None = None,\n    width: int | None = None,\n    height: int | None = None,\n    aspect: str | None = None,\n    tier: str | None = None,\n    is_edit: bool = False,\n    ref_dimensions: tuple[int, int] | None = None,\n) -> tuple[int, int]:\n    """Resolve output `(width, height)` aligned to 32 pixels."""\n    eff_tier = normalize_tier(tier, default="4k")\n    size_str = (size or "").strip()\n\n    if size_str and size_str.lower() != "auto":\n        m = re.fullmatch(r"(\\d+)\\s*[xX:]\\s*(\\d+)", size_str)\n        if not m:\n            raise ValueError(f"Invalid size \'{size}\': expected WIDTHxHEIGHT (e.g. 4096x4096, 2048x2048) or \'auto\'")\n        w, h = int(m.group(1)), int(m.group(2))\n    elif width is not None or height is not None:\n        if width is None or height is None:\n            raise ValueError("Both width and height must be provided together when size is omitted")\n        if isinstance(width, bool) or isinstance(height, bool):\n            raise ValueError("width and height must be integers")\n        w, h = int(width), int(height)\n    elif aspect:\n        asp = aspect.strip()\n        preset_map = ASPECT_PRESETS[eff_tier]\n        if asp not in preset_map:\n            valid = ", ".join(preset_map.keys())\n            raise ValueError(f"Unsupported aspect ratio \'{aspect}\': choose from {valid}")\n        return preset_map[asp]\n    elif is_edit and ref_dimensions and ref_dimensions[0] > 0 and ref_dimensions[1] > 0:\n        rw, rh = ref_dimensions\n        target_area = TIER_TARGET_AREAS[eff_tier]\n        ratio = rw / rh\n        # Snap to exact official preset if within 3% aspect ratio for native 4K/2K grid quality.\n        best_preset: tuple[int, int] | None = None\n        best_rel_diff = 0.03\n        for pw, ph in ASPECT_PRESETS[eff_tier].values():\n            rel_diff = abs((pw / ph) - ratio) / max(ratio, 1e-6)\n            if rel_diff < best_rel_diff:\n                best_rel_diff = rel_diff\n                best_preset = (pw, ph)\n        if best_preset is not None:\n            return best_preset\n        w = align_dim(int(math.sqrt(target_area * ratio)))\n        h = align_dim(int(math.sqrt(target_area / ratio)))\n        return w, h\n    else:\n        return ASPECT_PRESETS[eff_tier]["1:1"]\n\n    if w < MIN_DIM or h < MIN_DIM or w > MAX_DIM or h > MAX_DIM:\n        raise ValueError(f"Resolution {w}x{h} out of range [{MIN_DIM}, {MAX_DIM}]")\n    w = align_dim(w)\n    h = align_dim(h)\n    return w, h\n\n\ndef validate_min_generation_area(\n    value: Any,\n    *,\n    width: int,\n    height: int,\n    fast_preview: bool = False,\n) -> int | None:\n    if value is None:\n        if fast_preview and (width * height) < DEFAULT_MIN_GENERATION_AREA:\n            return max(MIN_GENERATION_AREA_BOUNDS[0], min(MIN_GENERATION_AREA_BOUNDS[1], width * height))\n        return None\n    if isinstance(value, bool) or not isinstance(value, int):\n        raise ValueError("min_generation_area must be an integer between 1024 and 16777216")\n    lo, hi = MIN_GENERATION_AREA_BOUNDS\n    if value < lo or value > hi:\n        raise ValueError(f"min_generation_area={value} out of allowed range [{lo}, {hi}]")\n    return value\n\n\ndef validate_h3_sampling_contract(payload: dict[str, Any]) -> None:\n    """Enforce MiniMax-H3-Image\'s strict 6-step PDD / single-positive-branch contract."""\n    steps = payload.get("num_inference_steps", payload.get("steps"))\n    if steps is not None and steps != "":\n        if isinstance(steps, bool) or not isinstance(steps, int) or steps != FIXED_PDD_STEPS:\n            raise ValueError(\n                f"MiniMax-H3-Image uses a fused {FIXED_PDD_STEPS}-step PDD schedule; "\n                f"num_inference_steps must be {FIXED_PDD_STEPS} or omitted (got {steps!r})"\n            )\n\n    guidance = payload.get("guidance_scale")\n    if guidance is not None and guidance != "":\n        if isinstance(guidance, bool) or not isinstance(guidance, (int, float)) or float(guidance) != 1.0:\n            raise ValueError(\n                f"MiniMax-H3-Image is CFG-distilled (single positive branch); "\n                f"guidance_scale must be 1 or omitted (got {guidance!r})"\n            )\n\n    neg = payload.get("negative_prompt")\n    if neg is not None and str(neg).strip() != "":\n        raise ValueError("MiniMax-H3-Image does not support negative_prompt (single positive branch)")\n\n    if payload.get("lora") or payload.get("lora_path"):\n        raise ValueError("MiniMax-H3-Image SGLang pipeline does not support LoRA adapters")\n\n\ndef inspect_image_bytes(data: bytes) -> tuple[int, int, str]:\n    """Return `(width, height, format_lower)` using Pillow (with EXIF transpose) or stdlib header parser."""\n    if not data:\n        raise ValueError("Empty image data")\n    try:\n        from PIL import Image, ImageOps  # type: ignore[import-not-found]\n\n        with Image.open(io.BytesIO(data)) as im:\n            fmt = (im.format or "PNG").lower()\n            im_oriented = ImageOps.exif_transpose(im)\n            w, h = im_oriented.size\n            if fmt == "jpg":\n                fmt = "jpeg"\n            return int(w), int(h), fmt\n    except ImportError:\n        pass\n\n    # Pure-stdlib fallback for PNG / WebP / JPEG headers in unit tests without Pillow.\n    if data.startswith(b"\\x89PNG\\r\\n\\x1a\\n") and len(data) >= 24:\n        w, h = struct.unpack(">II", data[16:24])\n        return int(w), int(h), "png"\n    if data.startswith(b"RIFF") and data[8:12] == b"WEBP" and len(data) >= 30:\n        chunk = data[12:16]\n        if chunk == b"VP8X" and len(data) >= 30:\n            w = 1 + int.from_bytes(data[24:27], "little")\n            h = 1 + int.from_bytes(data[27:30], "little")\n            return w, h, "webp"\n    if data.startswith(b"\\xff\\xd8"):\n        idx = 2\n        n = len(data)\n        while idx + 4 < n:\n            if data[idx] != 0xFF:\n                idx += 1\n                continue\n            marker = data[idx + 1]\n            while marker == 0xFF and idx + 2 < n:\n                idx += 1\n                marker = data[idx + 1]\n            if marker in (0xD8, 0xD9):\n                idx += 2\n                continue\n            if idx + 4 > n:\n                break\n            seg_len = struct.unpack(">H", data[idx + 2 : idx + 4])[0]\n            if marker in (0xC0, 0xC1, 0xC2) and idx + 9 <= n:\n                h, w = struct.unpack(">HH", data[idx + 5 : idx + 9])\n                return int(w), int(h), "jpeg"\n            idx += 2 + max(2, seg_len)\n    raise ValueError("Unsupported or corrupted image data")\n\n\ndef transcode_image_bytes(\n    src_bytes: bytes,\n    *,\n    output_format: str = "png",\n    output_quality: int = 92,\n    png_compress_level: int = 6,\n) -> tuple[bytes, int, int, str]:\n    """Encode/compress image bytes into `png`, `webp`, or `jpeg`, returning `(bytes, width, height, fmt)`."""\n    fmt = (output_format or "png").strip().lower()\n    if fmt == "jpg":\n        fmt = "jpeg"\n    if fmt not in ("png", "webp", "jpeg"):\n        raise ValueError(f"Unsupported output_format \'{output_format}\': expected png, webp, or jpeg")\n    quality = max(1, min(100, int(output_quality)))\n    compress_level = max(0, min(9, int(png_compress_level)))\n\n    try:\n        from PIL import Image  # type: ignore[import-not-found]\n\n        with Image.open(io.BytesIO(src_bytes)) as im:\n            w, h = im.size\n            src_fmt = (im.format or "").lower()\n            # Fast path: already PNG and caller doesn\'t need format change or recompression\n            if fmt == "png" and src_fmt == "png" and compress_level == 0:\n                return src_bytes, int(w), int(h), "png"\n            buf = io.BytesIO()\n            if fmt == "png":\n                im.save(buf, format="PNG", compress_level=compress_level)\n            elif fmt == "webp":\n                lossless = quality >= 100\n                im.save(buf, format="WEBP", quality=quality, lossless=lossless, method=4)\n            else:\n                rgb = im.convert("RGB") if im.mode not in ("RGB", "L") else im\n                rgb.save(buf, format="JPEG", quality=quality, optimize=True)\n            return buf.getvalue(), int(w), int(h), fmt\n    except ImportError:\n        w, h, detected = inspect_image_bytes(src_bytes)\n        if fmt != detected and not (fmt == "jpeg" and detected == "jpg"):\n            raise ValueError(f"Pillow is required to transcode {detected} -> {fmt}")\n        return src_bytes, w, h, fmt\n\n\ndef save_output_image_to_drive(\n    image_bytes: bytes,\n    spec: str,\n    *,\n    index: int = 0,\n    total: int = 1,\n    fmt: str = "png",\n    prefix: str = "h3",\n    data_dir: Path | None = None,\n    output_dir: Path | None = None,\n    mydrive_root: Path | None = None,\n    drive_root: Path | None = None,\n) -> tuple[str, str]:\n    """Persist generated image bytes strictly inside allowed Drive/data roots."""\n    eff_data = data_dir or DATA_DIR\n    eff_out = output_dir or DRIVE_OUTPUTS_DIR\n    eff_mydrive = mydrive_root or MYDRIVE_ROOT\n    eff_drive = drive_root or DRIVE_ROOT\n    eff_out.mkdir(parents=True, exist_ok=True)\n\n    ext = EXT_BY_FORMAT.get(fmt.lower(), ".png")\n    stamp = f"{time.strftime(\'%Y%m%d-%H%M%S\')}-{secrets.token_hex(3)}"\n    clean_spec = (spec or "").strip()\n\n    if clean_spec and clean_spec.lower() not in ("auto", "true", "1"):\n        dest = resolve_drive_path(\n            clean_spec,\n            data_dir=eff_data,\n            default_subdir="outputs",\n            mydrive_root=eff_mydrive,\n            drive_root=eff_drive,\n        )\n        if dest.is_dir() or clean_spec.endswith("/"):\n            dest.mkdir(parents=True, exist_ok=True)\n            suffix_idx = f"-{index + 1:02d}" if total > 1 else ""\n            dest = dest / f"{prefix}-{stamp}{suffix_idx}{ext}"\n        else:\n            dest.parent.mkdir(parents=True, exist_ok=True)\n            if not dest.suffix:\n                dest = dest.with_suffix(ext)\n            if total > 1:\n                dest = dest.with_name(f"{dest.stem}-{index + 1:02d}{dest.suffix}")\n    else:\n        suffix_idx = f"-{index + 1:02d}" if total > 1 else ""\n        dest = (eff_out / f"{prefix}-{stamp}{suffix_idx}{ext}").resolve()\n        roots = allowed_drive_roots(eff_data, eff_mydrive)\n        if not any(dest.is_relative_to(r) for r in roots):\n            raise ValueError("Resolved output path escaped allowed Drive roots")\n\n    tmp = dest.with_name(f".{dest.name}.{os.getpid()}.{secrets.token_hex(4)}.tmp")\n    try:\n        tmp.write_bytes(image_bytes)\n        os.replace(tmp, dest)\n    finally:\n        tmp.unlink(missing_ok=True)\n    return str(dest), rel_to_mydrive(dest, eff_mydrive)\n\n\ndef list_drive_image_files(\n    spec: str = "",\n    *,\n    kind: str = "images",\n    recursive: bool = True,\n    data_dir: Path | None = None,\n    mydrive_root: Path | None = None,\n    drive_root: Path | None = None,\n) -> list[dict[str, Any]]:\n    """List image files on mounted Google Drive strictly inside allowed roots."""\n    eff_data = data_dir or DATA_DIR\n    eff_mydrive = mydrive_root or MYDRIVE_ROOT\n    eff_drive = drive_root or DRIVE_ROOT\n    roots = allowed_drive_roots(eff_data, eff_mydrive)\n\n    default_sub: str | None = None\n    if not (spec or "").strip():\n        if kind == "outputs":\n            default_sub = "outputs"\n            (eff_data / "outputs").mkdir(parents=True, exist_ok=True)\n        elif kind == "inputs":\n            default_sub = "inputs"\n            (eff_data / "inputs").mkdir(parents=True, exist_ok=True)\n\n    target = resolve_drive_path(\n        spec or "",\n        data_dir=eff_data,\n        default_subdir=default_sub,\n        mydrive_root=eff_mydrive,\n        drive_root=eff_drive,\n    )\n    matched: list[Path] = []\n    if target.is_file():\n        if target.suffix.lower() in IMAGE_EXTS:\n            matched.append(target)\n    elif target.is_dir():\n        if not recursive:\n            for item in sorted(target.iterdir()):\n                rp = item.resolve()\n                if any(rp.is_relative_to(r) for r in roots) and rp.is_file() and rp.suffix.lower() in IMAGE_EXTS:\n                    matched.append(rp)\n        else:\n            skip = {".git", "__pycache__", "cache-archives", "kernel-archives", ".Trash", ".ipynb_checkpoints"}\n            for root_dir, dirnames, filenames in os.walk(target):\n                dirnames[:] = sorted(d for d in dirnames if d not in skip and not d.startswith("."))\n                for fn in sorted(filenames):\n                    rp = (Path(root_dir) / fn).resolve()\n                    if any(rp.is_relative_to(r) for r in roots) and rp.is_file() and rp.suffix.lower() in IMAGE_EXTS:\n                        matched.append(rp)\n\n    out: list[dict[str, Any]] = []\n    for p in matched:\n        st = p.stat()\n        out.append({\n            "drive_path": str(p),\n            "rel_path": rel_to_mydrive(p, eff_mydrive),\n            "name": p.name,\n            "type": "output" if "outputs" in p.parts else "image",\n            "size_bytes": st.st_size,\n            "size_mb": round(st.st_size / 1e6, 2),\n            "mtime": int(st.st_mtime),\n        })\n    return out\n\n\ndef store_upload_bytes(\n    data: bytes,\n    *,\n    filename: str = "image.png",\n    uploads_dir: Path | None = None,\n) -> dict[str, Any]:\n    """Content-address and persist a reference image upload on local SSD."""\n    if not data:\n        raise ValueError("Uploaded image body is empty")\n    if len(data) > MAX_UPLOAD_BYTES:\n        raise ValueError(f"Uploaded image exceeds {MAX_UPLOAD_BYTES // (1024 * 1024)} MiB limit")\n    w, h, fmt = inspect_image_bytes(data)\n    digest = hashlib.sha256(data).hexdigest()\n    eff_uploads = uploads_dir or UPLOADS_DIR\n    eff_uploads.mkdir(parents=True, exist_ok=True)\n    ext = EXT_BY_FORMAT.get(fmt, ".png")\n    dest = (eff_uploads / f"{digest}{ext}").resolve()\n    if not dest.is_relative_to(eff_uploads.resolve()):\n        raise ValueError("Invalid upload destination")\n    if not dest.is_file():\n        tmp = dest.with_name(f".{dest.name}.{os.getpid()}.{secrets.token_hex(4)}.tmp")\n        try:\n            tmp.write_bytes(data)\n            os.replace(tmp, dest)\n        finally:\n            tmp.unlink(missing_ok=True)\n    else:\n        os.utime(dest, None)\n    _prune_uploads_dir(eff_uploads)\n    return {\n        "upload_id": digest,\n        "uri": f"upload://{digest}",\n        "format": fmt,\n        "width": w,\n        "height": h,\n        "size_bytes": len(data),\n        "filename": Path(filename or "image.png").name,\n    }\n\n\ndef lookup_upload(upload_id: str, *, uploads_dir: Path | None = None) -> Path | None:\n    uid = (upload_id or "").strip().lower()\n    if not UPLOAD_ID_RE.fullmatch(uid):\n        raise ValueError(f"Invalid upload_id \'{upload_id}\': expected 64-char hex SHA-256")\n    eff_uploads = (uploads_dir or UPLOADS_DIR).resolve()\n    if not eff_uploads.is_dir():\n        return None\n    for ext in (".png", ".webp", ".jpg", ".jpeg"):\n        cand = (eff_uploads / f"{uid}{ext}").resolve()\n        if cand.is_relative_to(eff_uploads) and cand.is_file():\n            return cand\n    return None\n\n\ndef _prune_uploads_dir(uploads_dir: Path) -> None:\n    try:\n        files = [p for p in uploads_dir.iterdir() if p.is_file() and not p.name.startswith(".")]\n        total = sum(p.stat().st_size for p in files)\n        if total <= MAX_UPLOADS_CACHE_BYTES:\n            return\n        files.sort(key=lambda p: p.stat().st_mtime)\n        for p in files:\n            if total <= (MAX_UPLOADS_CACHE_BYTES * 3) // 4:\n                break\n            sz = p.stat().st_size\n            p.unlink(missing_ok=True)\n            total -= sz\n    except OSError:\n        pass\n\n\ndef resolve_reference_input(\n    spec: str,\n    *,\n    index: int,\n    data_dir: Path | None = None,\n    mydrive_root: Path | None = None,\n    drive_root: Path | None = None,\n    uploads_dir: Path | None = None,\n) -> tuple[str, bytes, str]:\n    """Resolve one reference image (`upload://`, `drive://`, `data:image/`, `https://`) -> `(filename, bytes, mime)`."""\n    raw = (spec or "").strip()\n    if not raw:\n        raise ValueError(f"Reference image #{index + 1} is empty")\n\n    if raw.startswith("upload://"):\n        uid = raw[len("upload://") :].strip()\n        up_path = lookup_upload(uid, uploads_dir=uploads_dir)\n        if up_path is None:\n            raise FileNotFoundError(f"Uploaded reference upload://{uid[:12]}... not found on server; re-upload required")\n        img_bytes = up_path.read_bytes()\n        _, _, fmt = inspect_image_bytes(img_bytes)\n        return up_path.name, img_bytes, MIME_BY_FORMAT.get(fmt, "image/png")\n\n    if raw.startswith("data:image/"):\n        if "," not in raw:\n            raise ValueError(f"Malformed data URI for reference image #{index + 1}")\n        header, b64_part = raw.split(",", 1)\n        img_bytes = base64.b64decode(b64_part)\n        _, _, fmt = inspect_image_bytes(img_bytes)\n        ext = EXT_BY_FORMAT.get(fmt, ".png")\n        mime = header[5:].split(";", 1)[0] or MIME_BY_FORMAT.get(fmt, "image/png")\n        return f"ref_{index + 1}{ext}", img_bytes, mime\n\n    if raw.startswith(("http://", "https://")):\n        parsed = urllib.parse.urlsplit(raw)\n        if parsed.hostname in ("127.0.0.1", "localhost", "::1", "169.254.169.254", "metadata.google.internal"):\n            raise ValueError(f"Refusing loopback/metadata URL in reference image #{index + 1}")\n        req = urllib.request.Request(raw, headers={"User-Agent": "colabllm-minimax-h3-image/1.0"})\n        with urllib.request.urlopen(req, timeout=60) as resp:\n            img_bytes = resp.read(MAX_UPLOAD_BYTES + 1)\n        if len(img_bytes) > MAX_UPLOAD_BYTES:\n            raise ValueError(f"Remote reference image #{index + 1} exceeds size limit")\n        _, _, fmt = inspect_image_bytes(img_bytes)\n        ext = EXT_BY_FORMAT.get(fmt, ".png")\n        return f"ref_{index + 1}{ext}", img_bytes, MIME_BY_FORMAT.get(fmt, "image/png")\n\n    # Otherwise resolve strictly inside allowed Google Drive / DATA_DIR roots (rejects /etc/passwd, traversal, etc.)\n    drive_file = resolve_drive_path(\n        raw,\n        data_dir=data_dir or DATA_DIR,\n        default_subdir="inputs",\n        mydrive_root=mydrive_root or MYDRIVE_ROOT,\n        drive_root=drive_root or DRIVE_ROOT,\n    )\n    if not drive_file.is_file():\n        raise FileNotFoundError(f"Google Drive reference image not found: {raw} (resolved: {drive_file})")\n    img_bytes = drive_file.read_bytes()\n    _, _, fmt = inspect_image_bytes(img_bytes)\n    mime = MIME_BY_FORMAT.get(fmt) or mimetypes.guess_type(str(drive_file))[0] or "image/png"\n    return drive_file.name, img_bytes, mime\n\n\ndef _parse_multipart_edits_request(content_type: str, body: bytes) -> dict[str, Any]:\n    """Parse standard OpenAI `multipart/form-data` `/v1/images/edits` into a canonical dict."""\n    if not content_type.lower().startswith("multipart/form-data"):\n        raise ValueError(f"Unsupported Content-Type \'{content_type}\' for /v1/images/edits")\n    raw_msg = f"Content-Type: {content_type}\\r\\nMIME-Version: 1.0\\r\\n\\r\\n".encode("utf-8") + body\n    msg = BytesParser(policy=email_default_policy).parsebytes(raw_msg)\n    if not msg.is_multipart():\n        raise ValueError("Invalid multipart/form-data payload")\n\n    fields: dict[str, Any] = {}\n    raw_files: list[tuple[str, bytes, str]] = []\n    url_refs: list[str] = []\n\n    for part in msg.iter_parts():\n        disp = part.get("Content-Disposition", "")\n        if "form-data" not in disp:\n            continue\n        name = part.get_param("name", header="content-disposition")\n        filename = part.get_param("filename", header="content-disposition")\n        payload_bytes = part.get_payload(decode=True) or b""\n        if not name:\n            continue\n        if filename is not None or name in ("image", "image[]"):\n            if filename is not None:\n                raw_files.append((str(filename or "image.png"), payload_bytes, part.get_content_type()))\n            else:\n                text_val = payload_bytes.decode("utf-8", errors="replace").strip()\n                if text_val:\n                    url_refs.append(text_val)\n        elif name in ("url", "url[]"):\n            text_val = payload_bytes.decode("utf-8", errors="replace").strip()\n            if text_val:\n                url_refs.append(text_val)\n        elif name == "mask":\n            raise ValueError("MiniMax-H3-Image does not support masked inpainting (\'mask\' field)")\n        else:\n            fields[name] = payload_bytes.decode("utf-8", errors="replace").strip()\n\n    for key in ("n", "seed", "num_inference_steps", "min_generation_area", "output_compression", "width", "height"):\n        if key in fields and fields[key] != "":\n            try:\n                fields[key] = int(fields[key])\n            except ValueError as exc:\n                raise ValueError(f"Field \'{key}\' must be an integer, got {fields[key]!r}") from exc\n    if "guidance_scale" in fields and fields["guidance_scale"] != "":\n        try:\n            fields["guidance_scale"] = float(fields["guidance_scale"])\n        except ValueError as exc:\n            raise ValueError(f"Field \'guidance_scale\' must be numeric, got {fields[\'guidance_scale\']!r}") from exc\n\n    fields["_multipart_files"] = raw_files\n    if url_refs:\n        fields["images"] = url_refs\n    return fields\n\n\ndef parse_byte_range(range_header: str | None, total_size: int) -> tuple[int, int] | None:\n    """Parse single HTTP `Range: bytes=start-end` header -> `(start, end_inclusive)` or `None`."""\n    if not range_header or total_size <= 0:\n        return None\n    raw = range_header.strip()\n    if not raw.lower().startswith("bytes=") or "," in raw:\n        return None\n    spec = raw[6:].strip()\n    if "-" not in spec:\n        return None\n    start_s, end_s = spec.split("-", 1)\n    start_s, end_s = start_s.strip(), end_s.strip()\n    if not start_s:\n        if not end_s.isdigit():\n            return None\n        suffix = int(end_s)\n        if suffix <= 0:\n            return None\n        start = max(0, total_size - suffix)\n        return start, total_size - 1\n    if not start_s.isdigit():\n        return None\n    start = int(start_s)\n    end = int(end_s) if end_s.isdigit() else (total_size - 1)\n    if start >= total_size or start < 0 or end < start:\n        raise ValueError(f"Unsatisfiable byte range {range_header} for size {total_size}")\n    return start, min(end, total_size - 1)\n\n\ndef default_sglang_caller(\n    task: str,\n    *,\n    prompt: str,\n    width: int,\n    height: int,\n    seed: int,\n    generator_device: str = "cpu",\n    min_generation_area: int | None = None,\n    quality: str | None = None,\n    references: list[tuple[str, bytes, str]] | None = None,\n    sglang_base_url: str | None = None,\n    sglang_output_dir: Path | None = None,\n    timeout: float = 900.0,\n) -> bytes:\n    """Invoke SGLang Diffusion on loopback using zero-Base64 File URL handoff when `--output-path` is active."""\n    base = (sglang_base_url or SGLANG_BASE_URL).rstrip("/")\n    out_root = (sglang_output_dir or SGLANG_OUTPUT_DIR).resolve()\n    use_url_handoff = out_root.is_dir()\n    resp_fmt = "url" if use_url_handoff else "b64_json"\n    size_str = f"{width}x{height}"\n\n    if task == "edit":\n        if not references:\n            raise ValueError("Edit task requires at least 1 reference image (<Picture 1>)")\n        form_fields: list[tuple[str, str]] = [\n            ("model", MODEL_REPO),\n            ("prompt", prompt),\n            ("size", size_str),\n            ("seed", str(int(seed))),\n            ("generator_device", generator_device),\n            ("num_inference_steps", str(FIXED_PDD_STEPS)),\n            ("response_format", resp_fmt),\n            ("output_format", "png"),\n            ("output_compression", "0"),\n        ]\n        if quality and quality != "auto":\n            form_fields.append(("quality", quality))\n        if min_generation_area is not None:\n            form_fields.append(("min_generation_area", str(int(min_generation_area))))\n        form_files = [("image", fname, mime, fbytes) for fname, fbytes, mime in references]\n        ct_hdr, raw_body = build_multipart_form(form_fields, form_files)\n        req = urllib.request.Request(\n            f"{base}/v1/images/edits",\n            data=raw_body,\n            headers={"Content-Type": ct_hdr},\n            method="POST",\n        )\n    else:\n        json_payload: dict[str, Any] = {\n            "model": MODEL_REPO,\n            "prompt": prompt,\n            "size": size_str,\n            "width": width,\n            "height": height,\n            "n": 1,\n            "seed": int(seed),\n            "generator_device": generator_device,\n            "num_inference_steps": FIXED_PDD_STEPS,\n            "response_format": resp_fmt,\n            "output_format": "png",\n            "output_compression": 0,\n        }\n        if quality and quality != "auto":\n            json_payload["quality"] = quality\n        if min_generation_area is not None:\n            json_payload["min_generation_area"] = int(min_generation_area)\n        req = urllib.request.Request(\n            f"{base}/v1/images/generations",\n            data=json.dumps(json_payload, ensure_ascii=False).encode("utf-8"),\n            headers={"Content-Type": "application/json"},\n            method="POST",\n        )\n\n    opener = urllib.request.build_opener(urllib.request.ProxyHandler({}))\n    try:\n        with opener.open(req, timeout=timeout) as resp:\n            resp_obj = json.loads(resp.read().decode("utf-8"))\n    except urllib.error.HTTPError as err:\n        detail = err.read().decode("utf-8", errors="replace")\n        raise RuntimeError(f"SGLang upstream HTTP {err.code}: {detail}") from err\n\n    items = resp_obj.get("data") or []\n    if not items or not isinstance(items[0], dict):\n        raise RuntimeError(f"SGLang returned empty image data: {resp_obj}")\n    first = items[0]\n\n    # Preferred fast path: read raw uncompressed PNG directly from shared local NVMe SSD and unlink scratch copy!\n    file_path = first.get("file_path")\n    if file_path:\n        fp = Path(str(file_path)).resolve()\n        if fp.is_relative_to(out_root) and fp.is_file():\n            try:\n                return fp.read_bytes()\n            finally:\n                fp.unlink(missing_ok=True)\n\n    if first.get("url"):\n        content_url = urllib.parse.urljoin(f"{base}/", str(first["url"]))\n        with opener.open(content_url, timeout=120) as c_resp:\n            return c_resp.read()\n\n    if first.get("b64_json"):\n        return base64.b64decode(first["b64_json"])\n\n    raise RuntimeError(f"SGLang response missing file_path, url, and b64_json: {first}")\n\n\nclass MiniMaxH3JobStore:\n    """Thread-safe / asyncio-friendly single-GPU job queue with local NVMe output retention."""\n\n    def __init__(\n        self,\n        *,\n        jobs_dir: Path | None = None,\n        uploads_dir: Path | None = None,\n        data_dir: Path | None = None,\n        drive_inputs_dir: Path | None = None,\n        drive_outputs_dir: Path | None = None,\n        mydrive_root: Path | None = None,\n        drive_root: Path | None = None,\n        sglang_caller: Callable[..., bytes] | None = None,\n    ) -> None:\n        self.jobs_dir = jobs_dir or JOBS_DIR\n        self.uploads_dir = uploads_dir or UPLOADS_DIR\n        self.data_dir = data_dir or DATA_DIR\n        self.drive_inputs_dir = drive_inputs_dir or (self.data_dir / "inputs" if data_dir else DRIVE_INPUTS_DIR)\n        self.drive_outputs_dir = drive_outputs_dir or (self.data_dir / "outputs" if data_dir else DRIVE_OUTPUTS_DIR)\n        self.mydrive_root = mydrive_root or MYDRIVE_ROOT\n        self.drive_root = drive_root or DRIVE_ROOT\n        self.sglang_caller = sglang_caller or default_sglang_caller\n        self.jobs: dict[str, dict[str, Any]] = {}\n        self._executor = ThreadPoolExecutor(max_workers=1, thread_name_prefix="h3-gpu-worker")\n\n    def prepare_canonical_job(self, payload: dict[str, Any], *, default_task: str = "t2i") -> dict[str, Any]:\n        if not isinstance(payload, dict):\n            raise ValueError("Request body must be a JSON object")\n        validate_h3_sampling_contract(payload)\n\n        raw_task = (payload.get("task") or payload.get("task_type") or default_task).strip().lower()\n        if raw_task in ("t2i", "text2image", "generation", "generations", "image"):\n            task = "t2i"\n        elif raw_task in ("edit", "edits", "i2i", "image_edit"):\n            task = "edit"\n        else:\n            raise ValueError(f"Unsupported task \'{raw_task}\': expected \'t2i\' or \'edit\'")\n\n        prompt = str(payload.get("prompt") or "").strip()\n        if not prompt:\n            raise ValueError("Field \'prompt\' must be a non-empty string")\n\n        multipart_files: list[tuple[str, bytes, str]] = list(payload.get("_multipart_files") or [])\n        raw_images = payload.get("images")\n        if raw_images is None and payload.get("image") is not None:\n            raw_images = [payload["image"]] if isinstance(payload["image"], str) else payload["image"]\n        image_specs: list[str] = []\n        if raw_images is not None:\n            if not isinstance(raw_images, list) or not all(isinstance(x, str) for x in raw_images):\n                raise ValueError("Field \'images\' must be a list of strings (drive://, upload://, https://, or data:)")\n            image_specs = [x.strip() for x in raw_images if x and x.strip()]\n\n        total_refs = len(multipart_files) + len(image_specs)\n        if task == "edit" and total_refs == 0:\n            raise ValueError("Edit task requires 1-16 reference images via \'images\' or multipart \'image\'")\n        if total_refs > MAX_REFERENCE_IMAGES:\n            raise ValueError(f"MiniMax-H3-Image supports at most {MAX_REFERENCE_IMAGES} reference images (got {total_refs})")\n        if total_refs > 0 and "task" not in payload and "task_type" not in payload:\n            task = "edit"\n\n        resolved_refs: list[tuple[str, bytes, str]] = []\n        for fname, fbytes, fmime in multipart_files:\n            inspect_image_bytes(fbytes)\n            resolved_refs.append((fname, fbytes, fmime or "image/png"))\n        for idx, spec in enumerate(image_specs):\n            resolved_refs.append(\n                resolve_reference_input(\n                    spec,\n                    index=len(resolved_refs) + idx,\n                    data_dir=self.data_dir,\n                    mydrive_root=self.mydrive_root,\n                    drive_root=self.drive_root,\n                    uploads_dir=self.uploads_dir,\n                )\n            )\n\n        if task == "edit":\n            prompt = normalize_edit_prompt_tags(prompt)\n\n        first_ref_dims: tuple[int, int] | None = None\n        if resolved_refs:\n            rw, rh, _ = inspect_image_bytes(resolved_refs[0][1])\n            first_ref_dims = (rw, rh)\n\n        width, height = resolve_dimensions(\n            size=payload.get("size"),\n            width=payload.get("width"),\n            height=payload.get("height"),\n            aspect=payload.get("aspect"),\n            tier=payload.get("tier"),\n            is_edit=(task == "edit"),\n            ref_dimensions=first_ref_dims,\n        )\n\n        min_area = validate_min_generation_area(\n            payload.get("min_generation_area"),\n            width=width,\n            height=height,\n            fast_preview=bool(payload.get("fast_preview", False)),\n        )\n\n        n_raw = payload.get("n", 1)\n        if n_raw is None:\n            n_raw = 1\n        if isinstance(n_raw, bool) or not isinstance(n_raw, int) or n_raw < 1 or n_raw > MAX_IMAGES_PER_JOB:\n            raise ValueError(f"Field \'n\' must be an integer between 1 and {MAX_IMAGES_PER_JOB}")\n\n        seed_raw = payload.get("seed")\n        if seed_raw is None or seed_raw == "":\n            seed = secrets.randbelow(2**31 - 1)\n        else:\n            if isinstance(seed_raw, bool) or not isinstance(seed_raw, int):\n                raise ValueError("Field \'seed\' must be an integer")\n            seed = int(seed_raw)\n\n        gen_dev = str(payload.get("generator_device") or "cpu").strip().lower()\n        if gen_dev not in ("cpu", "cuda"):\n            raise ValueError("generator_device must be \'cpu\' or \'cuda\'")\n\n        out_fmt = str(payload.get("output_format") or "png").strip().lower()\n        if out_fmt == "jpg":\n            out_fmt = "jpeg"\n        if out_fmt not in ("png", "webp", "jpeg"):\n            raise ValueError(f"Unsupported output_format \'{out_fmt}\': expected png, webp, or jpeg")\n\n        out_quality_raw = payload.get("output_quality", 92)\n        if isinstance(out_quality_raw, str) and out_quality_raw.lower() == "default":\n            out_quality = 92\n        else:\n            out_quality = max(1, min(100, int(out_quality_raw)))\n\n        png_level_raw = payload.get("png_compress_level", payload.get("output_compression", 6))\n        png_compress_level = 6 if png_level_raw is None else max(0, min(9, int(png_level_raw)))\n\n        save_spec = payload.get("save_to_drive")\n        if save_spec is not None and save_spec is not False and str(save_spec).strip() != "":\n            save_str = "auto" if save_spec is True else str(save_spec).strip()\n            # Pre-validate Drive destination path BEFORE running expensive GPU inference!\n            if save_str.lower() not in ("auto", "true", "1"):\n                resolve_drive_path(\n                    save_str,\n                    data_dir=self.data_dir,\n                    default_subdir="outputs",\n                    mydrive_root=self.mydrive_root,\n                    drive_root=self.drive_root,\n                )\n        else:\n            save_str = ""\n\n        quality_raw = str(payload.get("quality") or "").strip().lower()\n        if not quality_raw or quality_raw in ("auto", "lossless", "hd", "standard", "high", "medium", "low"):\n            quality: str | None = "lossless"\n        elif quality_raw == "exact":\n            quality = "exact"\n        else:\n            raise ValueError(f"Unsupported quality \'{quality_raw}\': expected \'lossless\' or \'exact\'")\n        return {\n            "model": normalize_model_name(payload.get("model")),\n            "task": task,\n            "prompt": prompt,\n            "width": width,\n            "height": height,\n            "size": f"{width}x{height}",\n            "n": n_raw,\n            "seed": seed,\n            "generator_device": gen_dev,\n            "min_generation_area": min_area,\n            "quality": quality,\n            "output_format": out_fmt,\n            "output_quality": out_quality,\n            "png_compress_level": png_compress_level,\n            "save_to_drive": save_str,\n            "drive_only": bool(payload.get("drive_only", False)),\n            "reference_count": len(resolved_refs),\n            "_references": resolved_refs,\n        }\n\n    def create_job(self, canonical: dict[str, Any]) -> dict[str, Any]:\n        job_id = f"job_{int(time.time())}_{secrets.token_hex(6)}"\n        # Ensure strict JOB_ID_RE compatibility: hex only after `job_`\n        job_id = f"job_{int(time.time()):08x}{secrets.token_hex(8)}"\n        now = time.time()\n        record: dict[str, Any] = {\n            "job_id": job_id,\n            "status": "queued",\n            "created_at": now,\n            "started_at": None,\n            "completed_at": None,\n            "elapsed_sec": 0.0,\n            "task": canonical["task"],\n            "model": canonical["model"],\n            "prompt": canonical["prompt"],\n            "size": canonical["size"],\n            "width": canonical["width"],\n            "height": canonical["height"],\n            "n": canonical["n"],\n            "seed": canonical["seed"],\n            "generator_device": canonical["generator_device"],\n            "min_generation_area": canonical["min_generation_area"],\n            "output_format": canonical["output_format"],\n            "output_quality": canonical["output_quality"],\n            "reference_count": canonical["reference_count"],\n            "save_to_drive": canonical["save_to_drive"] or None,\n            "completed_images": 0,\n            "total_images": canonical["n"],\n            "outputs": [],\n            "saved_drive_paths": [],\n            "rel_drive_paths": [],\n            "error": None,\n            "_cancelled": False,\n        }\n        self.jobs[job_id] = record\n        self._prune_jobs()\n        self._executor.submit(self._run_job_sync, job_id, canonical)\n        return self.public_view(record)\n\n    def run_sync(self, canonical: dict[str, Any]) -> tuple[dict[str, Any], list[bytes]]:\n        """Execute synchronously via the single-GPU worker and return `(job_view, encoded_images)`."""\n        job_id = f"job_{int(time.time()):08x}{secrets.token_hex(8)}"\n        now = time.time()\n        record: dict[str, Any] = {\n            "job_id": job_id,\n            "status": "queued",\n            "created_at": now,\n            "started_at": None,\n            "completed_at": None,\n            "elapsed_sec": 0.0,\n            "task": canonical["task"],\n            "model": canonical["model"],\n            "prompt": canonical["prompt"],\n            "size": canonical["size"],\n            "width": canonical["width"],\n            "height": canonical["height"],\n            "n": canonical["n"],\n            "seed": canonical["seed"],\n            "generator_device": canonical["generator_device"],\n            "min_generation_area": canonical["min_generation_area"],\n            "output_format": canonical["output_format"],\n            "output_quality": canonical["output_quality"],\n            "reference_count": canonical["reference_count"],\n            "save_to_drive": canonical["save_to_drive"] or None,\n            "completed_images": 0,\n            "total_images": canonical["n"],\n            "outputs": [],\n            "saved_drive_paths": [],\n            "rel_drive_paths": [],\n            "error": None,\n            "_cancelled": False,\n        }\n        self.jobs[job_id] = record\n        self._prune_jobs()\n        fut = self._executor.submit(self._run_job_sync, job_id, canonical)\n        encoded = fut.result()\n        if record["status"] != "succeeded":\n            raise RuntimeError(record.get("error") or "Image generation failed")\n        return self.public_view(record), encoded\n\n    def _run_job_sync(self, job_id: str, canonical: dict[str, Any]) -> list[bytes]:\n        record = self.jobs.get(job_id)\n        if record is None or record.get("_cancelled"):\n            if record is not None:\n                record["status"] = "cancelled"\n                record["completed_at"] = time.time()\n            return []\n\n        t0 = time.monotonic()\n        record["status"] = "running"\n        record["started_at"] = time.time()\n        job_dir = (self.jobs_dir / job_id).resolve()\n        job_dir.mkdir(parents=True, exist_ok=True)\n        encoded_list: list[bytes] = []\n\n        try:\n            total = canonical["n"]\n            for idx in range(total):\n                if record.get("_cancelled"):\n                    record["status"] = "cancelled"\n                    record["completed_at"] = time.time()\n                    record["elapsed_sec"] = round(time.monotonic() - t0, 3)\n                    return encoded_list\n\n                cur_seed = canonical["seed"] + idx\n                raw_png = self.sglang_caller(\n                    canonical["task"],\n                    prompt=canonical["prompt"],\n                    width=canonical["width"],\n                    height=canonical["height"],\n                    seed=cur_seed,\n                    generator_device=canonical["generator_device"],\n                    min_generation_area=canonical["min_generation_area"],\n                    quality=canonical["quality"],\n                    references=canonical["_references"] if canonical["task"] == "edit" else None,\n                )\n                out_bytes, actual_w, actual_h, actual_fmt = transcode_image_bytes(\n                    raw_png,\n                    output_format=canonical["output_format"],\n                    output_quality=canonical["output_quality"],\n                    png_compress_level=canonical["png_compress_level"],\n                )\n                ext = EXT_BY_FORMAT.get(actual_fmt, ".png")\n                local_file = (job_dir / f"output_{idx:02d}{ext}").resolve()\n                local_file.write_bytes(out_bytes)\n                sha = hashlib.sha256(out_bytes).hexdigest()\n\n                saved_drive = None\n                rel_drive = None\n                if canonical["save_to_drive"]:\n                    prefix = "h3-edit" if canonical["task"] == "edit" else "h3-gen"\n                    saved_drive, rel_drive = save_output_image_to_drive(\n                        out_bytes,\n                        canonical["save_to_drive"],\n                        index=idx,\n                        total=total,\n                        fmt=actual_fmt,\n                        prefix=prefix,\n                        data_dir=self.data_dir,\n                        output_dir=self.drive_outputs_dir,\n                        mydrive_root=self.mydrive_root,\n                        drive_root=self.drive_root,\n                    )\n                    record["saved_drive_paths"].append(saved_drive)\n                    record["rel_drive_paths"].append(rel_drive)\n\n                record["outputs"].append({\n                    "index": idx,\n                    "seed": cur_seed,\n                    "format": actual_fmt,\n                    "mime_type": MIME_BY_FORMAT.get(actual_fmt, "image/png"),\n                    "width": actual_w,\n                    "height": actual_h,\n                    "size_bytes": len(out_bytes),\n                    "sha256": sha,\n                    "content_path": f"/api/ct/jobs/{job_id}/content?index={idx}",\n                    "saved_drive_path": saved_drive,\n                    "rel_drive_path": rel_drive,\n                    "_local_file": str(local_file),\n                })\n                record["completed_images"] = idx + 1\n                record["elapsed_sec"] = round(time.monotonic() - t0, 3)\n                encoded_list.append(out_bytes)\n\n            record["status"] = "succeeded"\n            record["completed_at"] = time.time()\n            record["elapsed_sec"] = round(time.monotonic() - t0, 3)\n            return encoded_list\n        except Exception as exc:\n            record["status"] = "failed"\n            record["error"] = str(exc)\n            record["completed_at"] = time.time()\n            record["elapsed_sec"] = round(time.monotonic() - t0, 3)\n            return encoded_list\n        finally:\n            # Release reference image bytes from memory as soon as inference completes.\n            canonical["_references"] = []\n\n    def get_job(self, job_id: str) -> dict[str, Any] | None:\n        if not JOB_ID_RE.fullmatch(job_id or ""):\n            raise ValueError(f"Invalid job_id \'{job_id}\'")\n        rec = self.jobs.get(job_id)\n        if rec is None:\n            return None\n        if rec["status"] == "running" and rec.get("started_at"):\n            rec["elapsed_sec"] = round(max(0.0, time.time() - float(rec["started_at"])), 3)\n        return rec\n\n    def cancel_job(self, job_id: str) -> dict[str, Any] | None:\n        rec = self.get_job(job_id)\n        if rec is None:\n            return None\n        if rec["status"] == "queued":\n            rec["_cancelled"] = True\n            rec["status"] = "cancelled"\n            rec["completed_at"] = time.time()\n        elif rec["status"] == "running":\n            rec["_cancelled"] = True\n        return self.public_view(rec)\n\n    def get_output_bytes(\n        self,\n        job_id: str,\n        *,\n        index: int = 0,\n        output_format: str | None = None,\n        output_quality: int | None = None,\n    ) -> tuple[bytes, dict[str, Any]]:\n        rec = self.get_job(job_id)\n        if rec is None:\n            raise FileNotFoundError(f"Job \'{job_id}\' not found")\n        if rec["status"] != "succeeded":\n            raise ValueError(f"Job \'{job_id}\' is in state \'{rec[\'status\']}\' (not \'succeeded\')")\n        outputs = rec.get("outputs") or []\n        if index < 0 or index >= len(outputs):\n            raise ValueError(f"Output index {index} out of range (0..{max(0, len(outputs) - 1)})")\n        meta = dict(outputs[index])\n        local_p = Path(meta["_local_file"]).resolve()\n        if not local_p.is_relative_to(self.jobs_dir.resolve()) or not local_p.is_file():\n            raise FileNotFoundError(f"Output artifact for {job_id}[{index}] expired from local SSD")\n        raw = local_p.read_bytes()\n        req_fmt = (output_format or "").strip().lower()\n        if req_fmt == "jpg":\n            req_fmt = "jpeg"\n        if req_fmt and req_fmt != meta["format"]:\n            raw, w, h, fmt = transcode_image_bytes(\n                raw,\n                output_format=req_fmt,\n                output_quality=output_quality if output_quality is not None else 92,\n            )\n            meta = {\n                **meta,\n                "format": fmt,\n                "mime_type": MIME_BY_FORMAT.get(fmt, "image/png"),\n                "width": w,\n                "height": h,\n                "size_bytes": len(raw),\n                "sha256": hashlib.sha256(raw).hexdigest(),\n            }\n        meta.pop("_local_file", None)\n        return raw, meta\n\n    @staticmethod\n    def public_view(rec: dict[str, Any]) -> dict[str, Any]:\n        public_outputs = []\n        for item in rec.get("outputs") or []:\n            clean_item = {k: v for k, v in item.items() if not k.startswith("_")}\n            public_outputs.append(clean_item)\n        return {\n            k: (public_outputs if k == "outputs" else v)\n            for k, v in rec.items()\n            if not k.startswith("_")\n        }\n\n    def list_jobs(self, *, limit: int = 20) -> list[dict[str, Any]]:\n        ordered = sorted(self.jobs.values(), key=lambda r: float(r.get("created_at") or 0), reverse=True)\n        return [self.public_view(r) for r in ordered[: max(1, min(100, limit))]]\n\n    def _prune_jobs(self) -> None:\n        if len(self.jobs) <= MAX_RETAINED_JOBS:\n            return\n        terminal = [\n            r for r in self.jobs.values() if r.get("status") in ("succeeded", "failed", "cancelled")\n        ]\n        terminal.sort(key=lambda r: float(r.get("created_at") or 0))\n        while len(self.jobs) > MAX_RETAINED_JOBS and terminal:\n            victim = terminal.pop(0)\n            jid = victim["job_id"]\n            self.jobs.pop(jid, None)\n            jdir = (self.jobs_dir / jid).resolve()\n            if jdir.is_relative_to(self.jobs_dir.resolve()) and jdir.is_dir():\n                shutil.rmtree(jdir, ignore_errors=True)\n\n\ndef check_sglang_upstream_live(sglang_base_url: str | None = None, timeout: float = 5.0) -> bool:\n    base = (sglang_base_url or SGLANG_BASE_URL).rstrip("/")\n    opener = urllib.request.build_opener(urllib.request.ProxyHandler({}))\n    for route in ("/health", "/liveness", "/v1/models"):\n        try:\n            with opener.open(f"{base}{route}", timeout=timeout) as resp:\n                if resp.status == 200:\n                    return True\n        except Exception:\n            continue\n    return False\n\n\nclass MiniMaxH3ImageGateway:\n    """Pure-ASGI authenticated front-gateway for MiniMax-H3-Image."""\n\n    def __init__(\n        self,\n        *,\n        job_store: MiniMaxH3JobStore | None = None,\n        upstream_health_fn: Callable[[], bool] | None = None,\n    ) -> None:\n        self.store = job_store or MiniMaxH3JobStore()\n        self.upstream_health_fn = upstream_health_fn or check_sglang_upstream_live\n\n    async def __call__(self, scope: dict[str, Any], receive: Callable, send: Callable) -> None:\n        if scope.get("type") != "http":\n            return\n\n        method = (scope.get("method") or "GET").upper()\n        path = scope.get("path") or "/"\n        raw_qs = (scope.get("query_string") or b"").decode("utf-8", errors="replace")\n        query = urllib.parse.parse_qs(raw_qs, keep_blank_values=True)\n        headers = {\n            k.decode("latin1").lower(): v.decode("latin1")\n            for k, v in (scope.get("headers") or [])\n        }\n\n        if method == "OPTIONS":\n            await self._send_bytes(send, 204, b"", content_type="text/plain")\n            return\n\n        if not verify_bearer_auth(headers, API_TOKEN):\n            await self._send_json(\n                send,\n                401,\n                {"error": {"message": "Unauthorized: valid Bearer token or X-API-Key required", "type": "auth_error"}},\n                extra_headers=[(b"www-authenticate", b\'Bearer realm="colabllm"\')],\n            )\n            return\n\n        try:\n            if path == "/health" and method in ("GET", "HEAD"):\n                ok = await asyncio.to_thread(self.upstream_health_fn)\n                status = 200 if ok else 503\n                await self._send_json(\n                    send,\n                    status,\n                    {"status": "ok" if ok else "starting", "service": SERVICE_NAME, "model": MODEL_REPO},\n                    head_only=(method == "HEAD"),\n                )\n                return\n\n            if path == "/v1/models" and method == "GET":\n                created = int(time.time())\n                models = [\n                    {"id": MODEL_REPO, "object": "model", "created": created, "owned_by": "MiniMaxAI"},\n                ]\n                if MODEL_ALIAS and MODEL_ALIAS != MODEL_REPO:\n                    models.append({"id": MODEL_ALIAS, "object": "model", "created": created, "owned_by": "MiniMaxAI"})\n                await self._send_json(send, 200, {"object": "list", "data": models})\n                return\n\n            if path == "/api/ct/info" and method == "GET":\n                drive_mounted = self.store.mydrive_root.is_dir()\n                await self._send_json(\n                    send,\n                    200,\n                    {\n                        "service": SERVICE_NAME,\n                        "engine": "sglang-diffusion",\n                        "model_repo": MODEL_REPO,\n                        "served_models": [MODEL_REPO, *([MODEL_ALIAS] if MODEL_ALIAS != MODEL_REPO else [])],\n                        "quantization": QUANTIZATION,\n                        "placement": PLACEMENT,\n                        "warmup_mode": WARMUP_MODE,\n                        "pdd_steps": FIXED_PDD_STEPS,\n                        "default_min_generation_area": DEFAULT_MIN_GENERATION_AREA,\n                        "max_reference_images": MAX_REFERENCE_IMAGES,\n                        "max_images_per_job": MAX_IMAGES_PER_JOB,\n                        "aspect_presets": {\n                            tier: {asp: f"{w}x{h}" for asp, (w, h) in presets.items()}\n                            for tier, presets in ASPECT_PRESETS.items()\n                        },\n                        "drive_mounted": drive_mounted,\n                        "data_dir": str(self.store.data_dir),\n                        "drive_inputs_dir": str(self.store.drive_inputs_dir),\n                        "drive_outputs_dir": str(self.store.drive_outputs_dir),\n                        "internal_endpoints": [\n                            "POST /api/ct/uploads",\n                            "HEAD /api/ct/uploads/{sha256}",\n                            "POST /api/ct/jobs",\n                            "GET /api/ct/jobs",\n                            "GET /api/ct/jobs/{job_id}",\n                            "GET /api/ct/jobs/{job_id}/content",\n                            "DELETE /api/ct/jobs/{job_id}",\n                            "GET /api/ct/drive-list",\n                            "GET /api/ct/info",\n                        ],\n                    },\n                )\n                return\n\n            if path == "/api/ct/drive-list" and method == "GET":\n                spec = (query.get("path") or [""])[0]\n                kind = (query.get("kind") or ["images"])[0]\n                recursive = (query.get("recursive") or ["1"])[0] not in ("0", "false", "False")\n                files = await asyncio.to_thread(\n                    list_drive_image_files,\n                    spec,\n                    kind=kind,\n                    recursive=recursive,\n                    data_dir=self.store.data_dir,\n                    mydrive_root=self.store.mydrive_root,\n                    drive_root=self.store.drive_root,\n                )\n                await self._send_json(\n                    send,\n                    200,\n                    {\n                        "drive_mounted": self.store.mydrive_root.is_dir(),\n                        "data_dir": str(self.store.data_dir),\n                        "kind": kind,\n                        "files": files,\n                    },\n                )\n                return\n\n            # Content-addressed upload deduplication check & binary upload\n            if path.startswith("/api/ct/uploads/") and method in ("GET", "HEAD"):\n                uid = path[len("/api/ct/uploads/") :].strip("/")\n                found = lookup_upload(uid, uploads_dir=self.store.uploads_dir)\n                if found is None:\n                    await self._send_json(\n                        send,\n                        404,\n                        {"exists": False, "upload_id": uid},\n                        head_only=(method == "HEAD"),\n                    )\n                else:\n                    st = found.stat()\n                    await self._send_json(\n                        send,\n                        200,\n                        {\n                            "exists": True,\n                            "upload_id": uid,\n                            "uri": f"upload://{uid}",\n                            "size_bytes": st.st_size,\n                        },\n                        head_only=(method == "HEAD"),\n                    )\n                return\n\n            if path == "/api/ct/uploads" and method == "POST":\n                raw_body = await self._read_body(receive, max_bytes=MAX_UPLOAD_BYTES)\n                fname = headers.get("x-filename") or (query.get("filename") or ["image.png"])[0]\n                meta = await asyncio.to_thread(\n                    store_upload_bytes,\n                    raw_body,\n                    filename=fname,\n                    uploads_dir=self.store.uploads_dir,\n                )\n                await self._send_json(send, 200, meta)\n                return\n\n            # Lightweight async job API\n            if path == "/api/ct/jobs" and method == "GET":\n                limit = int((query.get("limit") or ["20"])[0])\n                await self._send_json(send, 200, {"jobs": self.store.list_jobs(limit=limit)})\n                return\n\n            if path == "/api/ct/jobs" and method == "POST":\n                raw_body = await self._read_body(receive, max_bytes=MAX_JSON_BODY_BYTES)\n                payload = json.loads(raw_body.decode("utf-8"))\n                canonical = await asyncio.to_thread(self.store.prepare_canonical_job, payload, default_task="t2i")\n                job_view = self.store.create_job(canonical)\n                await self._send_json(\n                    send,\n                    202,\n                    job_view,\n                    extra_headers=[(b"x-job-id", job_view["job_id"].encode("ascii"))],\n                )\n                return\n\n            if path.startswith("/api/ct/jobs/"):\n                sub = path[len("/api/ct/jobs/") :].strip("/")\n                parts = sub.split("/")\n                job_id = parts[0]\n\n                if len(parts) == 1 and method == "GET":\n                    rec = self.store.get_job(job_id)\n                    if rec is None:\n                        await self._send_json(send, 404, {"error": {"message": f"Job \'{job_id}\' not found"}})\n                        return\n                    await self._send_json(send, 200, self.store.public_view(rec))\n                    return\n\n                if len(parts) == 1 and method == "DELETE":\n                    cancelled = self.store.cancel_job(job_id)\n                    if cancelled is None:\n                        await self._send_json(send, 404, {"error": {"message": f"Job \'{job_id}\' not found"}})\n                        return\n                    await self._send_json(send, 200, cancelled)\n                    return\n\n                if len(parts) == 2 and parts[1] == "cancel" and method == "POST":\n                    cancelled = self.store.cancel_job(job_id)\n                    if cancelled is None:\n                        await self._send_json(send, 404, {"error": {"message": f"Job \'{job_id}\' not found"}})\n                        return\n                    await self._send_json(send, 200, cancelled)\n                    return\n\n                if len(parts) == 2 and parts[1] == "content" and method in ("GET", "HEAD"):\n                    idx = int((query.get("index") or ["0"])[0])\n                    req_fmt = (query.get("format") or [""])[0] or None\n                    req_qual_s = (query.get("quality") or [""])[0]\n                    req_qual = int(req_qual_s) if req_qual_s.isdigit() else None\n                    img_bytes, meta = await asyncio.to_thread(\n                        self.store.get_output_bytes,\n                        job_id,\n                        index=idx,\n                        output_format=req_fmt,\n                        output_quality=req_qual,\n                    )\n                    await self._stream_image_response(\n                        send,\n                        img_bytes,\n                        meta=meta,\n                        job_id=job_id,\n                        range_header=headers.get("range"),\n                        if_none_match=headers.get("if-none-match"),\n                        head_only=(method == "HEAD"),\n                    )\n                    return\n\n            # OpenAI-compatible synchronous endpoints (/v1/images/generations, /v1/images/edits, /api/ct/edit)\n            if path in ("/v1/images/generations", "/v1/images/edits", "/api/ct/edit") and method == "POST":\n                raw_body = await self._read_body(receive, max_bytes=MAX_JSON_BODY_BYTES)\n                ct = headers.get("content-type", "")\n                default_task = "t2i" if path == "/v1/images/generations" else "edit"\n                if ct.lower().startswith("multipart/form-data"):\n                    payload = await asyncio.to_thread(_parse_multipart_edits_request, ct, raw_body)\n                else:\n                    payload = json.loads(raw_body.decode("utf-8"))\n                resp_format = str(payload.get("response_format") or "b64_json").strip().lower()\n                if resp_format not in ("b64_json", "url"):\n                    raise ValueError(f"Unsupported response_format \'{resp_format}\': expected \'b64_json\' or \'url\'")\n\n                canonical = await asyncio.to_thread(\n                    self.store.prepare_canonical_job,\n                    payload,\n                    default_task=default_task,\n                )\n                job_view, encoded_list = await asyncio.to_thread(self.store.run_sync, canonical)\n                data_items: list[dict[str, Any]] = []\n                for idx, out_meta in enumerate(job_view.get("outputs") or []):\n                    item: dict[str, Any] = {"revised_prompt": canonical["prompt"]}\n                    if resp_format == "url":\n                        item["url"] = out_meta["content_path"]\n                    elif not canonical["drive_only"]:\n                        item["b64_json"] = base64.b64encode(encoded_list[idx]).decode("ascii")\n                    if out_meta.get("saved_drive_path"):\n                        item["saved_drive_path"] = out_meta["saved_drive_path"]\n                        item["rel_drive_path"] = out_meta["rel_drive_path"]\n                    data_items.append(item)\n\n                extra_hdrs: list[tuple[bytes, bytes]] = [(b"x-job-id", job_view["job_id"].encode("ascii"))]\n                if job_view.get("rel_drive_paths"):\n                    safe_dp = sanitize_header_param(job_view["rel_drive_paths"][0])\n                    extra_hdrs.append((b"x-saved-drive-path", safe_dp.encode("utf-8")))\n\n                await self._send_json(\n                    send,\n                    200,\n                    {\n                        "created": int(job_view["created_at"]),\n                        "job_id": job_view["job_id"],\n                        "size": job_view["size"],\n                        "output_format": job_view["output_format"],\n                        "elapsed_sec": job_view["elapsed_sec"],\n                        "saved_drive_paths": job_view["saved_drive_paths"],\n                        "rel_drive_paths": job_view["rel_drive_paths"],\n                        "data": data_items,\n                    },\n                    extra_headers=extra_hdrs,\n                )\n                return\n\n            if path == "/api/ct/logs" and method == "GET":\n                lines = max(1, min(200, int((query.get("lines") or ["40"])[0])))\n                await self._send_json(\n                    send,\n                    200,\n                    {\n                        "gateway_log": tail_service_log(LOG_FILE, lines=lines),\n                        "sglang_log": tail_service_log(SGLANG_LOG_FILE, lines=lines),\n                    },\n                )\n                return\n\n            await self._send_json(send, 404, {"error": {"message": f"Not found: {method} {path}"}})\n\n        except (ValueError, json.JSONDecodeError) as bad_req:\n            await self._send_json(send, 400, {"error": {"message": str(bad_req), "type": "invalid_request_error"}})\n        except FileNotFoundError as not_found:\n            await self._send_json(send, 404, {"error": {"message": str(not_found), "type": "not_found_error"}})\n        except Exception as err:\n            await self._send_json(send, 500, {"error": {"message": str(err), "type": "server_error"}})\n\n    @staticmethod\n    async def _read_body(receive: Callable, *, max_bytes: int) -> bytes:\n        chunks: list[bytes] = []\n        total = 0\n        while True:\n            msg = await receive()\n            mtype = msg.get("type")\n            if mtype == "http.request":\n                chunk = msg.get("body") or b""\n                total += len(chunk)\n                if total > max_bytes:\n                    raise ValueError(f"Request body exceeds {max_bytes // (1024 * 1024)} MiB limit")\n                chunks.append(chunk)\n                if not msg.get("more_body", False):\n                    break\n            elif mtype == "http.disconnect":\n                break\n        return b"".join(chunks)\n\n    @staticmethod\n    async def _send_bytes(\n        send: Callable,\n        status: int,\n        body: bytes,\n        *,\n        content_type: str = "application/octet-stream",\n        extra_headers: list[tuple[bytes, bytes]] | None = None,\n        head_only: bool = False,\n    ) -> None:\n        hdrs: list[tuple[bytes, bytes]] = [\n            (b"content-type", content_type.encode("utf-8")),\n            (b"content-length", str(len(body)).encode("ascii")),\n            *(extra_headers or []),\n        ]\n        await send({"type": "http.response.start", "status": status, "headers": _inject_cors_headers(hdrs)})\n        await send({"type": "http.response.body", "body": b"" if head_only else body, "more_body": False})\n\n    async def _send_json(\n        self,\n        send: Callable,\n        status: int,\n        payload: dict[str, Any],\n        *,\n        extra_headers: list[tuple[bytes, bytes]] | None = None,\n        head_only: bool = False,\n    ) -> None:\n        raw = json.dumps(payload, ensure_ascii=False).encode("utf-8")\n        await self._send_bytes(\n            send,\n            status,\n            raw,\n            content_type="application/json; charset=utf-8",\n            extra_headers=extra_headers,\n            head_only=head_only,\n        )\n\n    async def _stream_image_response(\n        self,\n        send: Callable,\n        img_bytes: bytes,\n        *,\n        meta: dict[str, Any],\n        job_id: str,\n        range_header: str | None,\n        if_none_match: str | None,\n        head_only: bool = False,\n    ) -> None:\n        etag = f\'"{meta["sha256"]}"\'\n        if if_none_match and if_none_match.strip() == etag:\n            await self._send_bytes(send, 304, b"", extra_headers=[(b"etag", etag.encode("ascii"))], head_only=True)\n            return\n\n        total = len(img_bytes)\n        ext = EXT_BY_FORMAT.get(meta["format"], ".png")\n        fname = sanitize_header_param(f"{job_id}_{meta[\'index\']:02d}{ext}")\n        base_headers: list[tuple[bytes, bytes]] = [\n            (b"accept-ranges", b"bytes"),\n            (b"etag", etag.encode("ascii")),\n            (b"x-job-id", job_id.encode("ascii")),\n            (b"x-image-width", str(meta["width"]).encode("ascii")),\n            (b"x-image-height", str(meta["height"]).encode("ascii")),\n            (b"x-image-format", str(meta["format"]).encode("ascii")),\n            (b"content-disposition", f\'inline; filename="{fname}"\'.encode("utf-8")),\n        ]\n        if meta.get("rel_drive_path"):\n            safe_dp = sanitize_header_param(str(meta["rel_drive_path"]))\n            base_headers.append((b"x-saved-drive-path", safe_dp.encode("utf-8")))\n\n        try:\n            byte_range = parse_byte_range(range_header, total)\n        except ValueError:\n            base_headers.append((b"content-range", f"bytes */{total}".encode("ascii")))\n            await self._send_bytes(send, 416, b"", extra_headers=base_headers, head_only=True)\n            return\n\n        if byte_range is not None:\n            start, end = byte_range\n            view = img_bytes[start : end + 1]\n            status = 206\n            base_headers.append((b"content-range", f"bytes {start}-{end}/{total}".encode("ascii")))\n        else:\n            view = img_bytes\n            status = 200\n\n        hdrs: list[tuple[bytes, bytes]] = [\n            (b"content-type", str(meta["mime_type"]).encode("ascii")),\n            (b"content-length", str(len(view)).encode("ascii")),\n            *base_headers,\n        ]\n        await send({"type": "http.response.start", "status": status, "headers": _inject_cors_headers(hdrs)})\n        if head_only or not view:\n            await send({"type": "http.response.body", "body": b"", "more_body": False})\n            return\n\n        offset = 0\n        n = len(view)\n        while offset < n:\n            nxt = min(n, offset + STREAM_CHUNK_BYTES)\n            await send({\n                "type": "http.response.body",\n                "body": view[offset:nxt],\n                "more_body": nxt < n,\n            })\n            offset = nxt\n\n\napp = MiniMaxH3ImageGateway()\n\n\ndef main() -> None:\n    import uvicorn  # type: ignore[import-not-found]\n\n    for d in (LOCAL_WORK_DIR, SGLANG_OUTPUT_DIR, SGLANG_INPUT_DIR, UPLOADS_DIR, JOBS_DIR):\n        d.mkdir(parents=True, exist_ok=True)\n    uvicorn.run(\n        app,\n        host="127.0.0.1",\n        port=SERVICE_PORT,\n        log_level="info",\n        timeout_keep_alive=75,\n    )\n\n\nif __name__ == "__main__":\n    main()\n', encoding="utf-8")

def build_sglang_args():
    args = [
        str(VENV_DIR / "bin" / "sglang"),
        "serve",
        "--model-path", str(MODEL_DIR),
        "--backend", "sglang",
        "--host", "127.0.0.1",
        "--port", str(SGLANG_PORT),
        "--output-path", str(SGLANG_OUTPUT_DIR),
        "--input-save-path", str(SGLANG_INPUT_DIR),
        "--minimax-h3-adaln-online", "true",
    ]
    # Placement policy tuned for Colab G4 (96 GB), H100/A100 (80 GB), and smaller GPUs
    if PLACEMENT == "resident":
        args.extend(["--component-residency", "all=resident"])
    elif PLACEMENT == "offload":
        args.extend(["--component-residency", "all=component-offload"])
    elif PLACEMENT == "layerwise":
        args.extend([
            "--component-residency", "all=layerwise-offload",
            "--layerwise-prefetch-size", "transformer=1,text_encoder=1,video_vae=1",
        ])
    else:
        if QUANTIZATION in ("fp8-all", "fp8", "mxfp8-all", "mxfp8") and GPU_VRAM_MIB >= 70000:
            args.extend(["--component-residency", "all=resident"])
        elif QUANTIZATION in ("fp8-dit", "mxfp8-dit") and GPU_VRAM_MIB >= 82000:
            args.extend(["--component-residency", "all=resident"])
        elif QUANTIZATION in ("bf16", "none", "native") and GPU_VRAM_MIB >= 92000:
            args.extend(["--component-residency", "all=resident"])
        elif GPU_VRAM_MIB >= 72000:
            args.extend(["--component-residency", "all=resident", "text_encoder=component-offload"])
        else:
            args.extend([
                "--component-residency", "all=layerwise-offload",
                "--layerwise-prefetch-size", "transformer=1,text_encoder=1,video_vae=1",
            ])

    if QUANTIZATION in ("fp8-all", "fp8"):
        args.extend(["--quantization", "fp8", "--component-quantizations.text_encoder", "fp8"])
    elif QUANTIZATION == "fp8-dit":
        args.extend(["--quantization", "fp8"])
    elif QUANTIZATION in ("mxfp8-all", "mxfp8"):
        args.extend(["--quantization", "mxfp8", "--component-quantizations.text_encoder", "mxfp8"])
    elif QUANTIZATION == "mxfp8-dit":
        args.extend(["--quantization", "mxfp8"])
    elif QUANTIZATION not in ("bf16", "none", "native"):
        raise ValueError(f"Unsupported CT_H3_IMAGE_QUANTIZATION={QUANTIZATION!r}")

    if WARMUP_MODE == "off":
        args.extend(["--warmup-mode", "off"])
    elif WARMUP_MODE in ("server", "request"):
        args.extend(["--warmup-mode", WARMUP_MODE])
    return args

def _common_runtime_env():
    env = dict(os.environ)
    venv_bin = str(VENV_DIR / "bin")
    cur_path = env.get("PATH") or "/usr/local/nvidia/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin"
    env.update({
        "VIRTUAL_ENV": str(VENV_DIR),
        "PATH": f"{venv_bin}:{cur_path}",
        "CT_API_TOKEN": API_TOKEN,
        "CT_SERVICE": SERVICE,
        "CT_SERVICE_PORT": str(SERVICE_PORT),
        "CT_H3_SGLANG_PORT": str(SGLANG_PORT),
        "CT_H3_SGLANG_URL": f"http://127.0.0.1:{SGLANG_PORT}",
        "CT_DATA_DIR": str(DATA_DIR),
        "CT_H3_LOCAL_WORK_DIR": str(H3_WORK),
        "CT_H3_IMAGE_MODEL": MODEL_REPO,
        "CT_H3_IMAGE_ALIAS": MODEL_ALIAS,
        "CT_H3_IMAGE_QUANTIZATION": QUANTIZATION,
        "CT_H3_IMAGE_PLACEMENT": PLACEMENT,
        "CT_H3_IMAGE_WARMUP": WARMUP_MODE,
        "FLASHINFER_WORKSPACE_BASE": str(FLASHINFER_CACHE_DIR),
        "TRITON_CACHE_DIR": str(TRITON_CACHE_DIR),
        "TORCHINDUCTOR_CACHE_DIR": str(INDUCTOR_CACHE_DIR),
        "SGLANG_JIT_CACHE_DIR": str(SGLANG_JIT_CACHE_DIR),
        "SGLANG_DG_CACHE_DIR": str(SGLANG_JIT_CACHE_DIR),
        "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
        "USE_TF": "0",
        "USE_FLAX": "0",
        "PYTHONPATH": f"{H3_WORK}:{env.get('PYTHONPATH', '')}".rstrip(":"),
    })
    return env

def start_sglang():
    if "sglang" in PROCS:
        ct_stop_proc(PROCS["sglang"][0])
    subprocess.run(["pkill", "-9", "-f", f"--port {SGLANG_PORT}"], check=False)
    time.sleep(1)
    cmd = build_sglang_args()
    h3log("Starting SGLang Diffusion: " + " ".join(cmd))
    proc = ct_spawn("sglang", cmd, env=_common_runtime_env(), cwd=str(H3_WORK))
    PROCS["sglang"] = (proc, start_sglang)
    return proc

def start_service():
    if not API_TOKEN:
        raise RuntimeError("CT_API_TOKEN is empty; refusing to start unauthenticated MiniMax-H3-Image gateway")
    if "sglang" not in PROCS or PROCS["sglang"][0].poll() is not None:
        start_sglang()
    if "service" in PROCS:
        ct_stop_proc(PROCS["service"][0])
    subprocess.run(["pkill", "-9", "-f", "minimax_h3_image_middleware.py"], check=False)
    time.sleep(1)
    proc = ct_spawn(
        "service",
        [str(VENV_DIR / "bin" / "python"), str(H3_WORK / "minimax_h3_image_middleware.py")],
        env=_common_runtime_env(),
        cwd=str(H3_WORK),
    )
    PROCS["service"] = (proc, start_service)
    return proc

start_sglang()
start_service()
h3log(f"Spawned SGLang (pid={PROCS['sglang'][0].pid} -> 127.0.0.1:{SGLANG_PORT}) + Gateway (pid={PROCS['service'][0].pid} -> 127.0.0.1:{SERVICE_PORT})")


In [ ]:
# @title 3. Auth layer -- Caddy: basic auth for browsers, `Authorization: Bearer <token>` for API clients
# PUBLIC_PORT must be free on the VM: Colab itself uses 8080 (runtime server) and 8012 (colab dfs shim).
# Only used when the service has no auth of its own (CT_AUTH=caddy). The service itself listens on
# 127.0.0.1 only, cloudflared targets Caddy, so every public request passes through auth.
REWRITE_HOST = True   # True: send Host: 127.0.0.1:<port> upstream (Ollama & co. reject foreign Host headers)
import platform
CADDY_ARCH = {"x86_64": "amd64", "aarch64": "arm64"}.get(platform.machine(), "amd64")

def start_caddy():
    if not os.path.exists("/usr/local/bin/caddy"):
        tag = json.loads(urllib.request.urlopen(
            "https://api.github.com/repos/caddyserver/caddy/releases/latest", timeout=30).read())["tag_name"]
        tgz = "/tmp/caddy.tgz"
        with urllib.request.urlopen(f"https://github.com/caddyserver/caddy/releases/download/{tag}/"
                                    f"caddy_{tag.lstrip('v')}_linux_{CADDY_ARCH}.tar.gz", timeout=60) as src, \
                open(tgz, "wb") as dst:
            shutil.copyfileobj(src, dst)
        subprocess.run(["tar", "-xzf", tgz, "-C", "/usr/local/bin", "caddy"], check=True)
    import hashlib
    cookie_tok = hashlib.sha256(f"ct-cookie-v2:{AUTH_USER}:{AUTH_PASS}:{API_TOKEN}".encode()).hexdigest()
    hashed = subprocess.check_output(["caddy", "hash-password", "--plaintext", AUTH_PASS], text=True).strip()
    upstream = f"""reverse_proxy 127.0.0.1:{SERVICE_PORT} {{
            {"header_up Host {upstream_hostport}" if REWRITE_HOST else ""}
            {"header_up Origin http://{upstream_hostport}" if REWRITE_HOST else ""}
            header_up -Authorization
            flush_interval -1
        }}"""
    caddyfile = f"""{{
    admin off
    auto_https off
}}
:{PUBLIC_PORT} {{
    encode zstd gzip
    @bearer header Authorization "Bearer {API_TOKEN}"
    handle @bearer {{
        {upstream}
    }}
    @cookie header Cookie *ct_session={cookie_tok}*
    handle @cookie {{
        {upstream}
    }}
    handle {{
        route {{
            basic_auth {{
                "{AUTH_USER}" {hashed}
            }}
            header +Set-Cookie "ct_session={cookie_tok}; Path=/; HttpOnly; SameSite=Lax"
            {upstream}
        }}
    }}
}}
"""
    open(f"{WORK}/Caddyfile", "w").write(caddyfile)
    subprocess.run(["pkill", "-f", "caddy run"], check=False)
    return ct_spawn("caddy", ["caddy", "run", "--config", f"{WORK}/Caddyfile", "--adapter", "caddyfile"])

AUTH_OK = False   # cell 4 refuses to open the tunnel unless this becomes True (fail closed)
if AUTH == "caddy":
    PROCS["caddy"] = (start_caddy(), start_caddy)
    time.sleep(2)
    if PROCS["caddy"][0].poll() is not None:   # e.g. port taken (Colab uses 8080 itself)
        raise RuntimeError(f"caddy exited -- is :{PUBLIC_PORT} in use?\n" + ct_tail("caddy"))
    # Must be 401 from *our* Caddy and must NOT leak Set-Cookie on unauthenticated 401 responses.
    ct_wait_http(f"http://127.0.0.1:{PUBLIC_PORT}/", timeout=60, ok=lambda c: c == 401)
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PUBLIC_PORT}/", timeout=5)
    except urllib.error.HTTPError as _e:
        if _e.headers.get("Set-Cookie"):
            raise RuntimeError("Caddy leaked Set-Cookie on unauthenticated 401 response!")
    AUTH_OK = True
    print(f"caddy on :{PUBLIC_PORT} -> 127.0.0.1:{SERVICE_PORT} (unauthenticated request -> 401)")
else:
    AUTH_OK = True
    print(f"auth={AUTH}: no proxy, cloudflared targets the service on :{PUBLIC_PORT} directly")


In [ ]:
# @title 4. Tunnel -- Cloudflare by default, or project-configured FRP + HTTPS
"""Self-contained Colab transport cell; injected into every service notebook."""
import hashlib
import json
import os
import platform
import re
import shutil
import signal
import subprocess
import tarfile
import tempfile
import threading
import time
import urllib.error
import urllib.request
from pathlib import Path


def render_frpc(config, local_port, ca_path):
    """TOML strings use JSON quoting, never shell interpolation."""
    return f'''serverAddr = {json.dumps(config['server_ip'])}
serverPort = {int(config['server_port'])}
loginFailExit = false
auth.method = "token"
auth.token = {json.dumps(config['token'])}
auth.additionalScopes = {json.dumps(config.get('auth_scopes',['HeartBeats','NewWorkConns']))}
transport.tls.enable = true
transport.tls.trustedCaFile = {json.dumps(str(ca_path))}
transport.tls.serverName = {json.dumps(config.get('server_name') or config['server_ip'])}
log.to = "console"
log.level = "info"
[[proxies]]
name = {json.dumps(config['proxy_name'])}
type = "tcp"
localIP = "127.0.0.1"
localPort = {int(local_port)}
remotePort = {int(config['remote_port'])}
'''


def install_frpc(destination, version):
    if destination.exists():
        return str(destination)
    arch={'x86_64':'amd64','aarch64':'arm64'}.get(platform.machine())
    if not arch:
        raise ValueError('Unsupported FRP client architecture')
    if not re.fullmatch(r'\d+\.\d+\.\d+',version) or tuple(map(int,version.split('.')))<(0,61,0):
        raise ValueError('Unsupported FRP client version; expected stable >= 0.61.0')
    asset=f'frp_{version}_linux_{arch}.tar.gz'
    base=f'https://github.com/fatedier/frp/releases/download/v{version}/'
    with tempfile.TemporaryDirectory() as tmp:
        archive=Path(tmp)/asset
        with urllib.request.urlopen(base+asset,timeout=60) as response, open(archive,'wb') as out:
            shutil.copyfileobj(response,out)
        checks=urllib.request.urlopen(base+'frp_sha256_checksums.txt',timeout=60).read().decode()
        expected=next((line.split()[0] for line in checks.splitlines()
                       if line.split() and line.split()[-1].lstrip('*')==asset),None)
        if expected is None or hashlib.sha256(archive.read_bytes()).hexdigest()!=expected:
            raise ValueError('FRP client checksum verification failed')
        with tarfile.open(archive) as tar:
            entry=tar.getmember(f'frp_{version}_linux_{arch}/frpc')
            if not entry.isfile():
                raise ValueError('Invalid FRP client binary')
            destination.parent.mkdir(parents=True,exist_ok=True)
            with tar.extractfile(entry) as source:
                destination.write_bytes(source.read())
            destination.chmod(0o755)
    return str(destination)


def run_tunnel(ctx):
    if not ctx.get('AUTH_OK'):
        raise RuntimeError('Auth layer not ready; refusing to expose the service')
    work=Path(ctx['WORK'])
    transport=os.environ.get('CT_TRANSPORT','cloudflare')
    if transport not in ('cloudflare','frp'):
        raise ValueError('Unsupported tunnel transport')
    # Stop only the tunnel previously started by this notebook (PID + proc
    # start time prevent accidental termination after PID reuse).
    marker=work/'tunnel-process.json'
    def identity(pid):
        try:
            stat=Path(f'/proc/{pid}/stat').read_text().rsplit(')',1)[1].split()
            return stat[19]
        except OSError:
            return None
    if marker.exists():
        previous=json.loads(marker.read_text())
        if previous.get('start_time') and identity(previous['pid'])==previous['start_time']:
            os.kill(previous['pid'],signal.SIGTERM)
            for _ in range(50):
                if identity(previous['pid']) is None:
                    break
                time.sleep(.1)
    def spawn(name,args):
        (work/(name+'.log')).write_text('')
        proc=ctx['ct_spawn'](name,args)
        marker.write_text(json.dumps(dict(pid=proc.pid,start_time=identity(proc.pid))))
        return proc

    if transport=='frp':
        config={key:os.environ['CT_FRP_'+key.upper()] for key in
                ('server_ip','server_port','remote_port','proxy_name','token')}
        config['server_name']=os.environ.get('CT_FRP_SERVER_NAME') or config['server_ip']
        config['auth_scopes']=json.loads(os.environ.get('CT_FRP_AUTH_SCOPES') or '["HeartBeats","NewWorkConns"]')
        public_host=ctx['PUBLIC_HOST']
        if not public_host:
            raise ValueError('FRP requires a public hostname')
        ca_path=work/'frp-ca.crt'
        ca_path.write_text(os.environ['CT_FRP_CA_CERT'])
        config_path=work/'frpc.toml'
        config_path.touch(mode=0o600,exist_ok=True)
        config_path.chmod(0o600)
        config_path.write_text(render_frpc(config,ctx['PUBLIC_PORT'],ca_path))
        binary=install_frpc(work/('frpc-'+os.environ['CT_FRP_VERSION']),os.environ['CT_FRP_VERSION'])
        result=subprocess.run([binary,'verify','-c',str(config_path)],capture_output=True)
        if result.returncode:
            raise RuntimeError('FRP client configuration validation failed')
        def start():
            return spawn('frpc',[binary,'-c',str(config_path)])
        key='frpc'
    else:
        cf_arch={'x86_64':'amd64','aarch64':'arm64'}.get(platform.machine(),'amd64')
        binary=ctx['ct_fetch_binary']('/usr/local/bin/cloudflared',
            f'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-{cf_arch}')
        def refresh_quick_url(proc_ref, wait_seconds=60):
            deadline_q=time.monotonic()+wait_seconds
            while time.monotonic()<deadline_q:
                if proc_ref.poll() is not None:
                    return None
                match_q=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',ctx['ct_tail']('cloudflared',100))
                if match_q:
                    new_url=match_q.group(0)
                    if new_url!=ctx.get('PUBLIC_URL'):
                        ctx['PUBLIC_URL']=new_url
                        print(f'CT_PUBLIC_URL={new_url}',flush=True)
                    return new_url
                time.sleep(1)
            return None
        ctx['ct_refresh_public_url']=refresh_quick_url
        def start():
            args=[binary,'tunnel','--no-autoupdate','--protocol','http2']
            if ctx['TUNNEL_TOKEN']:
                args+=['run','--token',ctx['TUNNEL_TOKEN']]
            else:
                args+=['--url',f'http://127.0.0.1:{ctx["PUBLIC_PORT"]}']
            proc_new=spawn('cloudflared',args)
            if not ctx['TUNNEL_TOKEN'] and ctx.get('PUBLIC_URL'):
                threading.Thread(target=refresh_quick_url,args=(proc_new,),daemon=True).start()
            return proc_new
        key='cloudflared'
    proc=start()
    ctx['PROCS'][key]=(proc,start)
    ctx['TUNNEL_PROCESS_KEY']=key
    deadline=time.monotonic()+180
    registered=False
    while time.monotonic()<deadline:
        if proc.poll() is not None:
            raise RuntimeError(f'{key} exited before registration; see /content/ct/{key}.log')
        log=ctx['ct_tail'](key,100)
        if transport=='frp':
            registered=registered or ('start proxy success' in log and config['proxy_name'] in log)
            if registered:
                url='https://'+ctx['PUBLIC_HOST']
                try:
                    with urllib.request.urlopen(url,timeout=5) as response:
                        if response.status<600:
                            break
                except urllib.error.HTTPError as error:
                    # A valid public TLS response proves the transport/site.
                    # 502 is expected while a large model is still loading;
                    # the service's keep-alive cell gates CT_READY on health.
                    code=error.code
                    error.close()
                    if code<600:
                        break
                except OSError:
                    pass
        elif ctx['TUNNEL_TOKEN'] and 'Registered tunnel connection' in log:
            url='https://'+ctx['PUBLIC_HOST']
            break
        elif not ctx['TUNNEL_TOKEN']:
            match=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',log)
            if match:
                url=match.group(0)
                break
        time.sleep(1)
    else:
        raise RuntimeError(f'{key} not ready; check tunnel logs, DNS, HTTPS and server ports')
    ctx['PUBLIC_URL']=url
    print(f'CT_PUBLIC_URL={url}',flush=True)

run_tunnel(globals())


In [ ]:
# @title 5. Readiness gate, page-cache release, compiler cache archiving & watchdog
import urllib.error
import urllib.request
if not globals().get("H3_SETUP_OK") or "service" not in PROCS or "sglang" not in PROCS:
    raise RuntimeError("MiniMax-H3-Image did not start because an earlier notebook cell failed; see traceback above.")

def http_json(url, token="", payload=None, timeout=20):
    headers = {"User-Agent": "Mozilla/5.0"}
    if token:
        headers["Authorization"] = f"Bearer {token}"
    data = None
    if payload is not None:
        headers["Content-Type"] = "application/json"
        data = json.dumps(payload).encode("utf-8")
    req = urllib.request.Request(url, data=data, headers=headers, method="POST" if data is not None else "GET")
    opener = urllib.request.build_opener(urllib.request.ProxyHandler({})) if "127.0.0.1" in url else urllib.request.build_opener()
    try:
        with opener.open(req, timeout=timeout) as r:
            body = r.read().decode("utf-8", errors="replace")
            return r.status, (json.loads(body) if body.strip().startswith("{") else {})
    except urllib.error.HTTPError as e:
        return e.code, {}

def auth_check(base, token):
    for candidate, expected in (("", 401), ("invalid-probe-token", 401), (token, 200)):
        code, _ = http_json(base + "/v1/models", candidate)
        if code != expected:
            raise RuntimeError(f"Auth test failed on {base}/v1/models: expected {expected}, got {code}")

LOCAL_BASE = f"http://127.0.0.1:{SERVICE_PORT}"
deadline = time.monotonic() + 3600
last_notice = 0
while time.monotonic() < deadline:
    if PROCS["sglang"][0].poll() is not None:
        raise RuntimeError("SGLang exited during startup:\n" + ct_tail("sglang", 90))
    if PROCS["service"][0].poll() is not None:
        raise RuntimeError("MiniMax-H3-Image gateway exited during startup:\n" + ct_tail("service", 60))
    try:
        code, _ = http_json(LOCAL_BASE + "/health", API_TOKEN, timeout=10)
        if code == 200:
            break
    except Exception:
        pass
    if time.monotonic() - last_notice >= 20:
        last_line = (ct_tail("sglang", 4).strip().splitlines() or ["loading 66.5B weights & warming PDD grids..."])[-1]
        h3log(f"Waiting for SGLang Diffusion | GPU: {ct_gpu_stat()} | {last_line[:110]}")
        last_notice = time.monotonic()
    time.sleep(5)
else:
    raise RuntimeError("Timed out waiting for SGLang Diffusion to become ready:\n" + ct_tail("sglang", 90))

auth_check(LOCAL_BASE, API_TOKEN)

# Drop 135 GB of cold .safetensors pages from Linux host page cache once resident in VRAM
if hasattr(os, "posix_fadvise") and hasattr(os, "POSIX_FADV_DONTNEED"):
    _dropped = 0
    for _sf in MODEL_DIR.rglob("*.safetensors"):
        try:
            _fd = os.open(str(_sf), os.O_RDONLY)
            try:
                os.posix_fadvise(_fd, 0, 0, os.POSIX_FADV_DONTNEED)
                _dropped += 1
            finally:
                os.close(_fd)
        except OSError:
            pass
    h3log(f"Released Linux page cache on {_dropped} safetensors shard(s)")

for _loc_c, _leg_c in CACHE_PAIRS:
    try:
        save_cache(_loc_c, CACHE_ARCHIVES, CACHE_KEY, h3log)
    except (OSError, ValueError) as _cerr:
        h3log(f"Cache archive save warning ({_loc_c.name}): {_cerr}")

ct_public_gate(
    PUBLIC_URL,
    lambda: http_json(PUBLIC_URL + "/v1/models", API_TOKEN, timeout=15)[0],
    lambda: auth_check(PUBLIC_URL, API_TOKEN),
    h3log,
    timeout=180,
    interval=8,
)
h3log(f"Public endpoint verified: {PUBLIC_URL}/v1/models")

print(f"{SERVICE} is up: {PUBLIC_URL} (model={MODEL_REPO}, quant={QUANTIZATION})")
print("CT_READY", flush=True)

ct_watchdog(
    lambda: http_json(LOCAL_BASE + "/health", API_TOKEN, timeout=10)[0] == 200,
    start_service,
    h3log,
    log_name="sglang",
    grace=1200,
    heartbeat=lambda up: h3log(f"heartbeat | up={up}s | GPU: {ct_gpu_stat()}"),
)
